In [ ]:
# Import standard libraries
from pathlib import Path
import sys
import warnings

# Import scientific computing libraries
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Add project root to path
sys.path.append("../")

# Import project-specific modules
from src.plots.continuous_maps import russia_continuous_multiplot
from src.timeseries_stats.trends import TrendAnalysis
from src.utils.logger import setup_logger

# Configure logger
logger = setup_logger(
    "forcings_final", log_file="../logs/forcings_final_analysis.log"
)

# Suppress warnings
warnings.filterwarnings("ignore")

# Configure matplotlib
plt.rcParams["font.family"] = "DeJavu Serif"
plt.rcParams["font.serif"] = ["Times New Roman"]


# Create wrapper functions for trend analysis
def mann_kendall_test(data: np.ndarray, alpha: float = 0.05) -> dict:
    """Wrapper for Mann-Kendall test."""
    # Create a simple series for the TrendAnalysis class
    series = pd.Series(
        data, index=pd.date_range("2000-01-01", periods=len(data), freq="ME")
    )
    analyzer = TrendAnalysis(series)
    result = analyzer.mann_kendall_test(alpha)
    # Standardize key names: z_statistic -> z_score for consistency
    if "z_statistic" in result:
        result["z_score"] = result["z_statistic"]
    return result


def sen_slope_estimator(data: np.ndarray) -> dict:
    """Wrapper for Sen's slope estimator."""
    series = pd.Series(
        data, index=pd.date_range("2000-01-01", periods=len(data), freq="ME")
    )
    analyzer = TrendAnalysis(series)
    mk_result = analyzer.mann_kendall_test()
    return {"sen_slope": mk_result.get("slope", np.nan)}


def linear_regression_trend(data: np.ndarray) -> dict:
    """Wrapper for linear regression trend with error handling."""
    series = pd.Series(
        data, index=pd.date_range("2000-01-01", periods=len(data), freq="ME")
    )
    analyzer = TrendAnalysis(series)
    try:
        return analyzer.linear_trend()
    except ValueError as e:
        # Handle case where all values are identical or other linear regression errors
        logger.warning(f"Linear regression failed: {e}")
        return {
            "slope": np.nan,
            "intercept": np.nan,
            "r_squared": np.nan,
            "p_value": np.nan,
        }


logger.info("Imports complete. Starting cross-entity analysis workflow.")
print("✓ Libraries loaded successfully")


# Meteorological Forcing Representation Analysis - CAMELS-RU

## Comprehensive Analysis of Precipitation Datasets and Water Balance

**Version**: 2.0 (Enhanced Analysis)  
**Date**: October 14, 2025  
**Analysis Period**: 2008-2023 (2008-2021 for GPCP)

---

### Analysis Objectives

1. **Dataset Comparison**: Compare ERA5-Land, MSWEP, and GPCP (2008-2021) precipitation products
2. **Temporal Trends**: Identify trends in precipitation and temperature independently
3. **Precipitation-Discharge Relationships**: Quantify correlations with systematic lag effects (0-7 days)
4. **Precipitation Sums Analysis**: Evaluate trends in annual and seasonal precipitation totals
5. **Water Balance Metrics**: Calculate runoff ratios, evapotranspiration proxies, and efficiency patterns
6. **Event-Based Analysis**: Assess precipitation-discharge response at event timescales
7. **Spatial Patterns**: Examine geographical coherence in forcing-response relationships
8. **Product Performance**: Assess which dataset best represents discharge variability

---

### Data Sources

- **Discharge**: CAMELS-RU daily discharge (2008-2023), high-quality gauges only
- **Precipitation**: 
  - ERA5-Land (2007-2023): ECMWF reanalysis, 0.1° resolution
  - MSWEP v2 (2007-2023): Multi-source merged product, 0.1° resolution
  - GPCP (2008-2021): Global Precipitation Climatology Project, 1.0° resolution
- **Temperature**: ERA5-Land daily temperature (min, max, mean) - trend analysis only
- **Geographical Clusters**: 15 clusters based on HydroATLAS physiographic attributes

---

### Key Metrics

**Precipitation Metrics:**
- Daily statistics and inter-dataset correlations
- Annual/seasonal/monthly precipitation sums
- Trend analysis (Mann-Kendall, Sen's slope) on daily and sum data

**Water Balance Metrics:**
- Runoff ratios (Q/P) - annual and seasonal
- Evapotranspiration proxies (P - Q)
- Water balance closure and residuals

**Precipitation-Discharge Relationships:**
- Pearson correlation with systematic lag analysis (0-7 days)
- Event-based response metrics
- Seasonal efficiency patterns

**Spatial Analysis:**
- Cluster-wise statistics
- Geographical pattern identification
- Dataset performance by region

---

### Notebook Structure

1. **Setup & Configuration** - Imports, logging, helper functions
2. **Data Loading & Preprocessing** - Load discharge, precipitation, clusters, spatial data
3. **Precipitation Dataset Comparison** - Statistics and inter-dataset correlations
4. **Temporal Trend Analysis** - Independent P and T trends
5. **Precipitation-Discharge Relationships** - Lag analysis and correlations
6. **Precipitation Sums Analysis** - Annual/seasonal totals and trends (NEW)
7. **Water Balance & Runoff Analysis** - Q/P ratios and ET proxies (NEW)
8. **Event-Based Response Analysis** - Precipitation events and discharge response (NEW)
9. **Seasonal Efficiency Patterns** - Season-specific P-Q relationships (NEW)
10. **Spatial Clustering & Geographic Patterns** - Merge datasets and cluster analysis
11. **Comprehensive Visualization** - All spatial maps and distributions
12. **Export Results & Documentation** - Save tables and generate reports

## 1. Setup and Data Loading

Import necessary libraries and load pre-computed analysis results.

### 1.1 Load Pre-Computed Discharge and Cluster Data

Load discharge quality metrics and geographical clustering information.

In [ ]:
# Load discharge data directories

discharge_dir_partial = Path("../data/HydroFiles/Discharge/partial/decent")
discharge_dir_full = Path("../data/HydroFiles/Discharge/full/decent")

# Load geographical cluster information
geo_clusters_df = pd.read_csv(
    "../results/geo_gauge_cluster_analysis_15_detailed.csv",
    index_col="gauge_id",
    dtype={"gauge_id": str},
)
geo_clusters_df.index = geo_clusters_df.index.astype(str)

# Load spatial data
gauge_gdf = gpd.read_file("../data/Geometry/GaugeGeomCAMELS.gpkg")
gauge_gdf.set_index("gauge_id", inplace=True)
gauge_gdf.index = gauge_gdf.index.astype(str)

ws_gdf = gpd.read_file("../data/Geometry/WatershedGeomCAMELS.gpkg")
ws_gdf.set_index("gauge_id", inplace=True)
ws_gdf.index = ws_gdf.index.astype(str)

basemap_data = gpd.read_file("../data/Geometry/basemap.gpkg")

# Get list of available gauges with discharge data
available_gauges = set()
for path in discharge_dir_partial.glob("*.csv"):
    available_gauges.add(path.stem)
for path in discharge_dir_full.glob("*.csv"):
    available_gauges.add(path.stem)

# Filter to gauges with both geometry and discharge data
valid_gauges = list(set(gauge_gdf.index) & available_gauges)

print(f"✓ Found {len(available_gauges)} gauges with discharge data")
print(f"✓ Found {len(valid_gauges)} gauges with both geometry and discharge")
print(f"✓ Loaded geographical clusters: {len(geo_clusters_df)} gauges")
print(f"✓ Loaded watershed geometries: {len(ws_gdf)} polygons")

logger.info(f"Initial data loaded: {len(valid_gauges)} valid gauges")

## 2. Precipitation Dataset Analysis

### 2.1 Load and Compare Precipitation Products

Load ERA5-Land, MSWEP, and GPCP precipitation data for all gauges.

In [ ]:
# Define precipitation data directories
era5_dir = Path("../data/MeteoData/CamelsRU/era5_land")
mswep_dir = Path("../data/MeteoData/CamelsRU/mswep")
gpcp_dir = Path("../data/MeteoData/CamelsRU/gpcp")

# Load precipitation data for all valid gauges
precipitation_data = {}
dataset_availability = {"era5": [], "mswep": [], "gpcp": []}

for gauge_id in valid_gauges:
    precipitation_data[gauge_id] = {}

    # ERA5-Land
    era5_path = era5_dir / f"{gauge_id}.csv"
    if era5_path.exists():
        try:
            df = pd.read_csv(era5_path, index_col="date", parse_dates=True)
            precipitation_data[gauge_id]["era5"] = df["prcp"]
            dataset_availability["era5"].append(gauge_id)
        except Exception as e:
            logger.warning(f"Error loading ERA5 for {gauge_id}: {e}")

    # MSWEP
    mswep_path = mswep_dir / f"{gauge_id}.csv"
    if mswep_path.exists():
        try:
            df = pd.read_csv(mswep_path, index_col="date", parse_dates=True)
            precipitation_data[gauge_id]["mswep"] = df["precipitation"].rename(
                "prcp"
            )
            dataset_availability["mswep"].append(gauge_id)
        except Exception as e:
            logger.warning(f"Error loading MSWEP for {gauge_id}: {e}")

    # GPCP (limited to 2021-09)
    gpcp_path = gpcp_dir / f"{gauge_id}.csv"
    if gpcp_path.exists():
        try:
            df = pd.read_csv(gpcp_path, index_col="date", parse_dates=True)
            precipitation_data[gauge_id]["gpcp"] = df["precip"]
            dataset_availability["gpcp"].append(gauge_id)
        except Exception as e:
            logger.warning(f"Error loading GPCP for {gauge_id}: {e}")

print("=" * 80)
print("PRECIPITATION DATASET AVAILABILITY")
print("=" * 80)
print(
    f"ERA5-Land:  {len(dataset_availability['era5'])} gauges ({len(dataset_availability['era5']) / len(valid_gauges) * 100:.1f}%)"
)
print(
    f"MSWEP:      {len(dataset_availability['mswep'])} gauges ({len(dataset_availability['mswep']) / len(valid_gauges) * 100:.1f}%)"
)
print(
    f"GPCP:       {len(dataset_availability['gpcp'])} gauges ({len(dataset_availability['gpcp']) / len(valid_gauges) * 100:.1f}%)"
)
print(
    f"All three:  {len(set(dataset_availability['era5']) & set(dataset_availability['mswep']) & set(dataset_availability['gpcp']))} gauges"
)
print("=" * 80)

logger.info(
    f"Loaded precipitation data: ERA5={len(dataset_availability['era5'])}, MSWEP={len(dataset_availability['mswep'])}, GPCP={len(dataset_availability['gpcp'])}"
)

### 2.2 Calculate Basic Precipitation Statistics

Calculate annual totals, variability, and inter-dataset comparisons.

In [ ]:
# Calculate precipitation statistics for each dataset
prcp_stats = []

for gauge_id in valid_gauges:
    gauge_stats = {"gauge_id": gauge_id}

    for dataset in ["era5", "mswep", "gpcp"]:
        if dataset in precipitation_data[gauge_id]:
            prcp = precipitation_data[gauge_id][dataset]

            # Basic statistics (2008-2023 for ERA5/MSWEP, 2008-2021 for GPCP)
            analysis_start = "2008-01-01"
            analysis_end = "2021-09-30" if dataset == "gpcp" else "2023-12-31"
            prcp_period = prcp.loc[analysis_start:analysis_end]

            if len(prcp_period) > 365:
                gauge_stats[f"{dataset}_mean_annual"] = float(
                    prcp_period.resample("YE").sum().mean()
                )
                gauge_stats[f"{dataset}_std_annual"] = float(
                    prcp_period.resample("YE").sum().std()
                )
                gauge_stats[f"{dataset}_cv_annual"] = (
                    gauge_stats[f"{dataset}_std_annual"]
                    / gauge_stats[f"{dataset}_mean_annual"]
                )
                gauge_stats[f"{dataset}_mean_daily"] = float(
                    prcp_period.mean()
                )
                gauge_stats[f"{dataset}_max_daily"] = float(prcp_period.max())
                gauge_stats[f"{dataset}_n_days"] = len(prcp_period)
                gauge_stats[f"{dataset}_n_wet_days"] = int(
                    (prcp_period > 1.0).sum()
                )
                gauge_stats[f"{dataset}_wet_day_fraction"] = gauge_stats[
                    f"{dataset}_n_wet_days"
                ] / len(prcp_period)

    if len(gauge_stats) > 1:  # Has at least one dataset
        prcp_stats.append(gauge_stats)

prcp_stats_df = pd.DataFrame(prcp_stats).set_index("gauge_id")

# Calculate inter-dataset correlations (for common period 2008-2021)
inter_dataset_corr = []

for gauge_id in valid_gauges:
    corr_stats = {"gauge_id": gauge_id}

    # ERA5 vs MSWEP
    if (
        "era5" in precipitation_data[gauge_id]
        and "mswep" in precipitation_data[gauge_id]
    ):
        era5 = precipitation_data[gauge_id]["era5"].loc["2008":"2021"]
        mswep = precipitation_data[gauge_id]["mswep"].loc["2008":"2021"]
        common_idx = era5.index.intersection(mswep.index)
        if len(common_idx) > 365:
            corr_stats["era5_mswep_r"] = float(
                era5.loc[common_idx].corr(mswep.loc[common_idx])
            )
            corr_stats["era5_mswep_bias"] = float(
                era5.loc[common_idx].mean() - mswep.loc[common_idx].mean()
            )

    # ERA5 vs GPCP
    if (
        "era5" in precipitation_data[gauge_id]
        and "gpcp" in precipitation_data[gauge_id]
    ):
        era5 = precipitation_data[gauge_id]["era5"].loc["2008":"2021-09"]
        gpcp = precipitation_data[gauge_id]["gpcp"]
        common_idx = era5.index.intersection(gpcp.index)
        if len(common_idx) > 365:
            corr_stats["era5_gpcp_r"] = float(
                era5.loc[common_idx].corr(gpcp.loc[common_idx])
            )
            corr_stats["era5_gpcp_bias"] = float(
                era5.loc[common_idx].mean() - gpcp.loc[common_idx].mean()
            )

    # MSWEP vs GPCP
    if (
        "mswep" in precipitation_data[gauge_id]
        and "gpcp" in precipitation_data[gauge_id]
    ):
        mswep = precipitation_data[gauge_id]["mswep"].loc["2008":"2021-09"]
        gpcp = precipitation_data[gauge_id]["gpcp"]
        common_idx = mswep.index.intersection(gpcp.index)
        if len(common_idx) > 365:
            corr_stats["mswep_gpcp_r"] = float(
                mswep.loc[common_idx].corr(gpcp.loc[common_idx])
            )
            corr_stats["mswep_gpcp_bias"] = float(
                mswep.loc[common_idx].mean() - gpcp.loc[common_idx].mean()
            )

    if len(corr_stats) > 1:
        inter_dataset_corr.append(corr_stats)

inter_dataset_corr_df = pd.DataFrame(inter_dataset_corr).set_index("gauge_id")

print("=" * 80)
print("PRECIPITATION DATASET STATISTICS SUMMARY")
print("=" * 80)
print(f"\nDataset statistics calculated for {len(prcp_stats_df)} gauges")
print(
    f"Inter-dataset correlations calculated for {len(inter_dataset_corr_df)} gauges"
)
print("\nMean annual precipitation (mm/yr):")
for dataset in ["era5", "mswep", "gpcp"]:
    col = f"{dataset}_mean_annual"
    if col in prcp_stats_df.columns:
        print(
            f"  {dataset.upper():8s}: {prcp_stats_df[col].mean():.1f} ± {prcp_stats_df[col].std():.1f}"
        )
print("\nInter-dataset correlations (daily values):")
for pair in ["era5_mswep", "era5_gpcp", "mswep_gpcp"]:
    col = f"{pair}_r"
    if col in inter_dataset_corr_df.columns:
        print(
            f"  {pair.upper():15s}: {inter_dataset_corr_df[col].mean():.3f} ± {inter_dataset_corr_df[col].std():.3f}"
        )
print("=" * 80)

logger.info(
    f"Precipitation statistics calculated for {len(prcp_stats_df)} gauges"
)

### 2.3 Precipitation Temporal Trends (Independent Analysis)

Calculate trends in precipitation for each dataset using Mann-Kendall test.

In [ ]:
# Calculate precipitation trends for each dataset
prcp_trends = []

for gauge_id in valid_gauges:
    trend_stats = {"gauge_id": gauge_id}

    for dataset in ["era5", "mswep", "gpcp"]:
        if dataset in precipitation_data[gauge_id]:
            prcp = precipitation_data[gauge_id][dataset]

            # Use appropriate period
            analysis_start = "2008-01-01"
            analysis_end = "2021-09-30" if dataset == "gpcp" else "2023-12-31"
            prcp_period = prcp.loc[analysis_start:analysis_end]

            if len(prcp_period) > 365:
                # Aggregate to monthly for trend analysis (more robust)
                monthly_prcp = prcp_period.resample("ME").sum()

                if len(monthly_prcp) >= 24:  # At least 2 years
                    # Mann-Kendall test
                    mk_result = mann_kendall_test(monthly_prcp.values)
                    trend_stats[f"{dataset}_mk_trend"] = mk_result["trend"]
                    trend_stats[f"{dataset}_mk_p_value"] = mk_result["p_value"]
                    trend_stats[f"{dataset}_mk_z_score"] = mk_result["z_score"]

                    # Sen's slope (mm/month per year)
                    sen_result = sen_slope_estimator(monthly_prcp.values)
                    trend_stats[f"{dataset}_sen_slope"] = (
                        sen_result["sen_slope"] * 12
                    )  # Convert to mm/yr per year

                    # Linear regression
                    lr_result = linear_regression_trend(monthly_prcp.values)
                    trend_stats[f"{dataset}_lr_slope"] = (
                        lr_result["slope"] * 12
                    )
                    trend_stats[f"{dataset}_lr_r_squared"] = lr_result[
                        "r_squared"
                    ]

    if len(trend_stats) > 1:
        prcp_trends.append(trend_stats)

prcp_trends_df = pd.DataFrame(prcp_trends).set_index("gauge_id")

# Summary statistics
print("=" * 80)
print("PRECIPITATION TREND ANALYSIS SUMMARY")
print("=" * 80)

for dataset in ["era5", "mswep", "gpcp"]:
    mk_col = f"{dataset}_mk_trend"
    pval_col = f"{dataset}_mk_p_value"
    slope_col = f"{dataset}_sen_slope"

    if mk_col in prcp_trends_df.columns:
        total = prcp_trends_df[mk_col].notna().sum()
        significant = (prcp_trends_df[pval_col] < 0.05).sum()
        # Use string comparison for trend direction
        increasing = (
            (prcp_trends_df[pval_col] < 0.05)
            & (prcp_trends_df[mk_col] == "increasing")
        ).sum()
        decreasing = (
            (prcp_trends_df[pval_col] < 0.05)
            & (prcp_trends_df[mk_col] == "decreasing")
        ).sum()
        mean_slope = prcp_trends_df[slope_col].mean()
        median_slope = prcp_trends_df[slope_col].median()

        print(f"\n{dataset.upper()} Precipitation Trends:")
        print(f"  Total gauges:       {total}")
        print(
            f"  Significant trends: {significant} ({100 * significant / total:.1f}%)"
        )
        print(
            f"  Increasing:         {increasing} ({100 * increasing / total:.1f}%)"
        )
        print(
            f"  Decreasing:         {decreasing} ({100 * decreasing / total:.1f}%)"
        )
        print(f"  Mean Sen's slope:   {mean_slope:.2f} mm/yr per year")
        print(f"  Median Sen's slope: {median_slope:.2f} mm/yr per year")

print("=" * 80)

logger.info(
    f"Precipitation trends calculated for {len(prcp_trends_df)} gauges"
)


### 2.4 Seasonal Precipitation Trends Analysis

Analyze precipitation trends by season to identify seasonal patterns in precipitation changes.

In [ ]:
# Calculate seasonal precipitation trends for ERA5-Land (most complete dataset)
prcp_seasonal_trends = []

for gauge_id in valid_gauges:
    if "era5" not in precipitation_data[gauge_id]:
        continue

    seasonal_stats = {"gauge_id": gauge_id}
    prcp = precipitation_data[gauge_id]["era5"]
    prcp_period = prcp.loc["2008":"2023"]

    if len(prcp_period) > 365:
        # Add season column
        prcp_df = pd.DataFrame({"prcp": prcp_period})
        prcp_df["season"] = prcp_df.index.month.map(assign_season)

        # Analyze trends for each season
        for season in ["winter", "spring", "summer", "autumn"]:
            season_data = prcp_df[prcp_df["season"] == season]["prcp"]

            if len(season_data) > 90:  # At least ~3 months per year
                # Monthly aggregation for trend analysis
                monthly_season = season_data.resample("ME").sum()

                if len(monthly_season) >= 12:  # At least 4 years * 3 months
                    # Mann-Kendall test
                    mk_result = mann_kendall_test(monthly_season.values)
                    seasonal_stats[f"prcp_{season}_mk_trend"] = mk_result[
                        "trend"
                    ]
                    seasonal_stats[f"prcp_{season}_mk_p_value"] = mk_result[
                        "p_value"
                    ]
                    seasonal_stats[f"prcp_{season}_mk_z_score"] = mk_result[
                        "z_score"
                    ]

                    # Sen's slope (mm/month per year)
                    sen_result = sen_slope_estimator(monthly_season.values)
                    seasonal_stats[f"prcp_{season}_sen_slope"] = (
                        sen_result["sen_slope"] * 12
                    )

                    # Linear regression
                    lr_result = linear_regression_trend(monthly_season.values)
                    seasonal_stats[f"prcp_{season}_lr_slope"] = (
                        lr_result["slope"] * 12
                    )
                    seasonal_stats[f"prcp_{season}_lr_r_squared"] = lr_result[
                        "r_squared"
                    ]

    if len(seasonal_stats) > 1:
        prcp_seasonal_trends.append(seasonal_stats)

prcp_seasonal_trends_df = pd.DataFrame(prcp_seasonal_trends).set_index(
    "gauge_id"
)

# Summary statistics
print("=" * 80)
print("SEASONAL PRECIPITATION TREND ANALYSIS (ERA5-Land, 2008-2023)")
print("=" * 80)

for season in ["winter", "spring", "summer", "autumn"]:
    mk_col = f"prcp_{season}_mk_trend"
    pval_col = f"prcp_{season}_mk_p_value"
    slope_col = f"prcp_{season}_sen_slope"

    if mk_col in prcp_seasonal_trends_df.columns:
        total = prcp_seasonal_trends_df[mk_col].notna().sum()
        significant = (prcp_seasonal_trends_df[pval_col] < 0.05).sum()
        increasing = (
            (prcp_seasonal_trends_df[pval_col] < 0.05)
            & (prcp_seasonal_trends_df[mk_col] == "increasing")
        ).sum()
        decreasing = (
            (prcp_seasonal_trends_df[pval_col] < 0.05)
            & (prcp_seasonal_trends_df[mk_col] == "decreasing")
        ).sum()
        mean_slope = prcp_seasonal_trends_df[slope_col].mean()
        median_slope = prcp_seasonal_trends_df[slope_col].median()

        print(f"\n{season.capitalize()} Precipitation:")
        print(f"  Total gauges:       {total}")
        print(
            f"  Significant trends: {significant} ({100 * significant / total:.1f}%)"
        )
        print(
            f"  Increasing:         {increasing} ({100 * increasing / total:.1f}%)"
        )
        print(
            f"  Decreasing:         {decreasing} ({100 * decreasing / total:.1f}%)"
        )
        print(f"  Mean Sen's slope:   {mean_slope:.2f} mm/yr²")
        print(f"  Median Sen's slope: {median_slope:.2f} mm/yr²")

print("=" * 80)

logger.info(
    f"Seasonal precipitation trends calculated for {len(prcp_seasonal_trends_df)} gauges"
)

## 3. Temperature Temporal Trends (Independent Analysis)

### 3.1 Calculate Temperature Trends

Analyze temperature trends separately from precipitation-discharge relationships.

In [ ]:
# Load temperature data from ERA5-Land
temperature_data = {}

for gauge_id in valid_gauges:
    era5_path = era5_dir / f"{gauge_id}.csv"
    if era5_path.exists():
        try:
            df = pd.read_csv(era5_path, index_col="date", parse_dates=True)
            # Temperature values are already in daily degrees Celsius
            temperature_data[gauge_id] = {
                "t_mean": df["t_mean"],
                "t_min": df["t_min"],
                "t_max": df["t_max"],
            }
        except Exception as e:
            logger.warning(f"Error loading temperature for {gauge_id}: {e}")


# Helper function to assign seasons
def assign_season(month: int) -> str:
    """Assign meteorological season based on month."""
    if month in [12, 1, 2]:
        return "winter"
    elif month in [3, 4, 5]:
        return "spring"
    elif month in [6, 7, 8]:
        return "summer"
    else:  # 9, 10, 11
        return "autumn"


# Calculate temperature trends by season
temp_trends = []

for gauge_id in valid_gauges:
    if gauge_id not in temperature_data:
        continue

    trend_stats = {"gauge_id": gauge_id}

    for var in ["t_mean", "t_min", "t_max"]:
        temp_series = temperature_data[gauge_id][var].loc["2008":"2023"]

        if len(temp_series) > 365:
            # Add season column
            temp_df = pd.DataFrame({var: temp_series})
            temp_df["season"] = temp_df.index.month.map(assign_season)

            # Analyze trends for each season
            for season in ["winter", "spring", "summer", "autumn"]:
                season_data = temp_df[temp_df["season"] == season][var]

                if len(season_data) > 90:  # At least ~3 months per year
                    # Monthly aggregation for trend analysis
                    monthly_season = season_data.resample("ME").mean()

                    if (
                        len(monthly_season) >= 12
                    ):  # At least 4 years * 3 months
                        # Mann-Kendall test
                        mk_result = mann_kendall_test(monthly_season.values)
                        trend_stats[f"{var}_{season}_mk_trend"] = mk_result[
                            "trend"
                        ]
                        trend_stats[f"{var}_{season}_mk_p_value"] = mk_result[
                            "p_value"
                        ]
                        trend_stats[f"{var}_{season}_mk_z_score"] = mk_result[
                            "z_score"
                        ]

                        # Sen's slope (°C per year)
                        sen_result = sen_slope_estimator(monthly_season.values)
                        trend_stats[f"{var}_{season}_sen_slope"] = (
                            sen_result["sen_slope"] * 12
                        )

                        # Linear regression
                        lr_result = linear_regression_trend(
                            monthly_season.values
                        )
                        trend_stats[f"{var}_{season}_lr_slope"] = (
                            lr_result["slope"] * 12
                        )
                        trend_stats[f"{var}_{season}_lr_r_squared"] = (
                            lr_result["r_squared"]
                        )

            # Also calculate annual (all seasons combined) trends
            monthly_temp = temp_series.resample("ME").mean()

            if len(monthly_temp) >= 24:
                mk_result = mann_kendall_test(monthly_temp.values)
                trend_stats[f"{var}_annual_mk_trend"] = mk_result["trend"]
                trend_stats[f"{var}_annual_mk_p_value"] = mk_result["p_value"]
                trend_stats[f"{var}_annual_mk_z_score"] = mk_result["z_score"]

                sen_result = sen_slope_estimator(monthly_temp.values)
                trend_stats[f"{var}_annual_sen_slope"] = (
                    sen_result["sen_slope"] * 12
                )

                lr_result = linear_regression_trend(monthly_temp.values)
                trend_stats[f"{var}_annual_lr_slope"] = lr_result["slope"] * 12
                trend_stats[f"{var}_annual_lr_r_squared"] = lr_result[
                    "r_squared"
                ]

    if len(trend_stats) > 1:
        temp_trends.append(trend_stats)

temp_trends_df = pd.DataFrame(temp_trends).set_index("gauge_id")

# Summary statistics
print("=" * 80)
print("TEMPERATURE TREND ANALYSIS SUMMARY (2008-2023)")
print("SEASONAL AND ANNUAL TRENDS")
print("=" * 80)

for var in ["t_mean", "t_min", "t_max"]:
    var_label = {
        "t_mean": "Mean Temperature",
        "t_min": "Min Temperature",
        "t_max": "Max Temperature",
    }[var]

    print(f"\n{var_label}:")
    print("-" * 80)

    # Annual trends
    mk_col = f"{var}_annual_mk_trend"
    pval_col = f"{var}_annual_mk_p_value"
    slope_col = f"{var}_annual_sen_slope"

    if mk_col in temp_trends_df.columns:
        total = temp_trends_df[mk_col].notna().sum()
        significant = (temp_trends_df[pval_col] < 0.05).sum()
        warming = (
            (temp_trends_df[pval_col] < 0.05)
            & (temp_trends_df[mk_col] == "increasing")
        ).sum()
        cooling = (
            (temp_trends_df[pval_col] < 0.05)
            & (temp_trends_df[mk_col] == "decreasing")
        ).sum()
        mean_slope = temp_trends_df[slope_col].mean()
        median_slope = temp_trends_df[slope_col].median()

        print("  Annual:")
        print(f"    Total gauges:       {total}")
        print(
            f"    Significant trends: {significant} ({100 * significant / total:.1f}%)"
        )
        print(
            f"    Warming:            {warming} ({100 * warming / total:.1f}%)"
        )
        print(
            f"    Cooling:            {cooling} ({100 * cooling / total:.1f}%)"
        )
        print(f"    Mean Sen's slope:   {mean_slope:.4f} °C/year")
        print(f"    Median Sen's slope: {median_slope:.4f} °C/year")

    # Seasonal trends
    for season in ["winter", "spring", "summer", "autumn"]:
        mk_col = f"{var}_{season}_mk_trend"
        pval_col = f"{var}_{season}_mk_p_value"
        slope_col = f"{var}_{season}_sen_slope"

        if mk_col in temp_trends_df.columns:
            total = temp_trends_df[mk_col].notna().sum()
            significant = (temp_trends_df[pval_col] < 0.05).sum()
            warming = (
                (temp_trends_df[pval_col] < 0.05)
                & (temp_trends_df[mk_col] == "increasing")
            ).sum()
            cooling = (
                (temp_trends_df[pval_col] < 0.05)
                & (temp_trends_df[mk_col] == "decreasing")
            ).sum()
            mean_slope = temp_trends_df[slope_col].mean()
            median_slope = temp_trends_df[slope_col].median()

            print(f"  {season.capitalize()}:")
            print(f"    Total gauges:       {total}")
            print(
                f"    Significant trends: {significant} ({100 * significant / total:.1f}%)"
            )
            print(
                f"    Warming:            {warming} ({100 * warming / total:.1f}%)"
            )
            print(
                f"    Cooling:            {cooling} ({100 * cooling / total:.1f}%)"
            )
            print(f"    Mean Sen's slope:   {mean_slope:.4f} °C/year")
            print(f"    Median Sen's slope: {median_slope:.4f} °C/year")

print("=" * 80)

logger.info(
    f"Seasonal temperature trends calculated for {len(temp_trends_df)} gauges"
)


## 4. Precipitation-Discharge Correlation Analysis with Lag Effects

### 4.1 Load Discharge Data

Load discharge time series for correlation analysis.

In [ ]:
# Load discharge data
discharge_data = {}

for gauge_id in valid_gauges:
    # Try partial first, then full
    discharge_path_partial = discharge_dir_partial / f"{gauge_id}.csv"
    discharge_path_full = discharge_dir_full / f"{gauge_id}.csv"

    discharge_path = None
    if discharge_path_partial.exists():
        discharge_path = discharge_path_partial
    elif discharge_path_full.exists():
        discharge_path = discharge_path_full

    if discharge_path:
        try:
            df = pd.read_csv(
                discharge_path, index_col="date", parse_dates=True
            )
            if "q_mm_day" in df.columns:
                discharge_data[gauge_id] = df["q_mm_day"]
        except Exception as e:
            logger.warning(f"Error loading discharge for {gauge_id}: {e}")

print(f"✓ Loaded discharge data for {len(discharge_data)} gauges")
logger.info(f"Discharge data loaded for {len(discharge_data)} gauges")

### 4.2 Calculate Precipitation-Discharge Correlations with Systematic Lag Analysis (1-7 days)

Systematically test lags from 0 to 7 days for each precipitation dataset.

## 5. Precipitation Sums Analysis

### Objective
Analyze trends in annual and seasonal precipitation totals to understand changes in total water input to catchments. This complements daily precipitation trend analysis by focusing on cumulative water availability.

### Analysis Components
1. Calculate annual and seasonal precipitation sums for each dataset
2. Assess inter-annual variability (CV, standard deviation)
3. Trend analysis on sums using Mann-Kendall and Sen's slope
4. Compare sum trends across ERA5-Land, MSWEP, and GPCP
5. Spatial patterns in precipitation sum changes
6. Statistical significance and dataset agreement

### Key Questions
- Are annual precipitation totals increasing or decreasing?
- Do seasonal accumulation patterns show coherent trends?
- How do sum trends differ from daily precipitation trends?
- Which datasets agree on total water input changes?

### 5.1 Calculate Annual and Seasonal Precipitation Sums

In [ ]:
# Calculate annual and seasonal precipitation sums
prcp_sums_data = []


# Define seasons
def get_season(month):
    """Map month to meteorological season."""
    if month in [12, 1, 2]:
        return "DJF"  # Winter
    elif month in [3, 4, 5]:
        return "MAM"  # Spring
    elif month in [6, 7, 8]:
        return "JJA"  # Summer
    else:
        return "SON"  # Autumn


print("=" * 80)
print("PRECIPITATION SUMS ANALYSIS")
print("=" * 80)
print("Calculating annual and seasonal precipitation totals...")

for gauge_id in valid_gauges:
    if gauge_id not in precipitation_data:
        continue

    sums_stats = {"gauge_id": gauge_id}

    for dataset in ["era5", "mswep", "gpcp"]:
        if dataset not in precipitation_data[gauge_id]:
            continue

        prcp = precipitation_data[gauge_id][dataset]

        # Set analysis period (GPCP ends earlier)
        analysis_end = "2021-09-30" if dataset == "gpcp" else "2023-12-31"
        prcp_period = prcp.loc["2008":analysis_end]

        if len(prcp_period) < 365:
            continue

        # Annual sums
        annual_sums = prcp_period.resample("YE").sum()
        sums_stats[f"{dataset}_annual_sum_mean"] = annual_sums.mean()
        sums_stats[f"{dataset}_annual_sum_std"] = annual_sums.std()
        sums_stats[f"{dataset}_annual_sum_cv"] = (
            annual_sums.std() / annual_sums.mean()
            if annual_sums.mean() > 0
            else np.nan
        )
        sums_stats[f"{dataset}_annual_sum_min"] = annual_sums.min()
        sums_stats[f"{dataset}_annual_sum_max"] = annual_sums.max()

        # Seasonal sums - calculate for each year
        prcp_with_season = prcp_period.to_frame("precip")
        prcp_with_season["year"] = prcp_with_season.index.year
        prcp_with_season["season"] = prcp_with_season.index.month.map(
            get_season
        )

        for season in ["DJF", "MAM", "JJA", "SON"]:
            seasonal_data = prcp_with_season[
                prcp_with_season["season"] == season
            ]
            # Group by year and sum
            seasonal_sums = seasonal_data.groupby("year")["precip"].sum()

            if len(seasonal_sums) > 0:
                sums_stats[f"{dataset}_{season}_sum_mean"] = (
                    seasonal_sums.mean()
                )
                sums_stats[f"{dataset}_{season}_sum_std"] = seasonal_sums.std()

    if len(sums_stats) > 1:
        prcp_sums_data.append(sums_stats)

prcp_sums_df = pd.DataFrame(prcp_sums_data).set_index("gauge_id")

print(f"✓ Calculated precipitation sums for {len(prcp_sums_df)} gauges")
print("\nAnnual Sum Statistics (all datasets, mm/yr):")
print(
    f"  ERA5-Land:  {prcp_sums_df['era5_annual_sum_mean'].mean():.1f} ± {prcp_sums_df['era5_annual_sum_std'].mean():.1f}"
)
print(
    f"  MSWEP:      {prcp_sums_df['mswep_annual_sum_mean'].mean():.1f} ± {prcp_sums_df['mswep_annual_sum_std'].mean():.1f}"
)
if "gpcp_annual_sum_mean" in prcp_sums_df.columns:
    print(
        f"  GPCP:       {prcp_sums_df['gpcp_annual_sum_mean'].mean():.1f} ± {prcp_sums_df['gpcp_annual_sum_std'].mean():.1f}"
    )

print("\nSeasonal Sum Statistics (ERA5-Land, mm/season):")
for season in ["DJF", "MAM", "JJA", "SON"]:
    col = f"era5_{season}_sum_mean"
    if col in prcp_sums_df.columns:
        print(
            f"  {season}: {prcp_sums_df[col].mean():.1f} ± {prcp_sums_df[col].std():.1f}"
        )

logger.info(f"Precipitation sums calculated for {len(prcp_sums_df)} gauges")

### 5.2 Trend Analysis on Annual Precipitation Sums

In [ ]:
# Trend analysis on annual precipitation sums
prcp_sum_trends = []

print("\n" + "=" * 80)
print("TREND ANALYSIS ON PRECIPITATION SUMS")
print("=" * 80)
print("Analyzing trends in annual and seasonal precipitation totals...")

for gauge_id in valid_gauges:
    if gauge_id not in precipitation_data:
        continue

    trend_stats = {"gauge_id": gauge_id}

    for dataset in ["era5", "mswep", "gpcp"]:
        if dataset not in precipitation_data[gauge_id]:
            continue

        prcp = precipitation_data[gauge_id][dataset]
        analysis_end = "2021-09-30" if dataset == "gpcp" else "2023-12-31"
        prcp_period = prcp.loc["2008":analysis_end]

        if len(prcp_period) < 365:
            continue

        # Annual sum trend
        annual_sums = prcp_period.resample("YE").sum()

        if len(annual_sums) >= 10:  # Need sufficient years for trend
            mk_result = mann_kendall_test(annual_sums.values)
            sen_result = sen_slope_estimator(annual_sums.values)
            lr_result = linear_regression_trend(annual_sums.values)

            trend_stats[f"{dataset}_annual_sum_mk_trend"] = mk_result.get(
                "trend", "no trend"
            )
            trend_stats[f"{dataset}_annual_sum_mk_p_value"] = mk_result.get(
                "p", np.nan
            )
            trend_stats[f"{dataset}_annual_sum_mk_z_score"] = mk_result.get(
                "z_score", np.nan
            )
            trend_stats[f"{dataset}_annual_sum_sen_slope"] = sen_result.get(
                "sen_slope", np.nan
            )
            trend_stats[f"{dataset}_annual_sum_lr_slope"] = lr_result.get(
                "slope", np.nan
            )
            trend_stats[f"{dataset}_annual_sum_lr_r_squared"] = lr_result.get(
                "r_squared", np.nan
            )

        # Seasonal sum trends
        prcp_with_season = prcp_period.to_frame("precip")
        prcp_with_season["year"] = prcp_with_season.index.year
        prcp_with_season["season"] = prcp_with_season.index.month.map(
            get_season
        )

        for season in ["DJF", "MAM", "JJA", "SON"]:
            seasonal_data = prcp_with_season[
                prcp_with_season["season"] == season
            ]
            seasonal_sums = seasonal_data.groupby("year")["precip"].sum()

            if len(seasonal_sums) >= 10:
                mk_result = mann_kendall_test(seasonal_sums.values)
                sen_result = sen_slope_estimator(seasonal_sums.values)

                trend_stats[f"{dataset}_{season}_sum_mk_trend"] = (
                    mk_result.get("trend", "no trend")
                )
                trend_stats[f"{dataset}_{season}_sum_sen_slope"] = (
                    sen_result.get("sen_slope", np.nan)
                )

    if len(trend_stats) > 1:
        prcp_sum_trends.append(trend_stats)

prcp_sum_trends_df = pd.DataFrame(prcp_sum_trends).set_index("gauge_id")

# Summary statistics
print(f"✓ Trend analysis completed for {len(prcp_sum_trends_df)} gauges\n")

for dataset in ["era5", "mswep", "gpcp"]:
    col = f"{dataset}_annual_sum_mk_trend"
    if col in prcp_sum_trends_df.columns:
        total = prcp_sum_trends_df[col].notna().sum()
        significant = (
            prcp_sum_trends_df[f"{dataset}_annual_sum_mk_p_value"] < 0.05
        ).sum()
        increasing = (
            (prcp_sum_trends_df[f"{dataset}_annual_sum_mk_p_value"] < 0.05)
            & (prcp_sum_trends_df[col] == "increasing")
        ).sum()
        decreasing = (
            (prcp_sum_trends_df[f"{dataset}_annual_sum_mk_p_value"] < 0.05)
            & (prcp_sum_trends_df[col] == "decreasing")
        ).sum()
        mean_slope = prcp_sum_trends_df[
            f"{dataset}_annual_sum_sen_slope"
        ].mean()

        print(f"{dataset.upper()} Annual Sum Trends:")
        print(f"  Total gauges:        {total}")
        print(
            f"  Significant (α=0.05): {significant} ({100 * significant / total:.1f}%)"
        )
        print(
            f"  Increasing:          {increasing} ({100 * increasing / total:.1f}%)"
        )
        print(
            f"  Decreasing:          {decreasing} ({100 * decreasing / total:.1f}%)"
        )
        print(f"  Mean Sen's slope:    {mean_slope:.2f} mm/yr²")
        print()

logger.info(
    f"Precipitation sum trends analyzed for {len(prcp_sum_trends_df)} gauges"
)

## 6. Water Balance & Runoff Ratio Analysis

### Objective
Quantify the precipitation-to-runoff conversion efficiency and assess water balance closure across catchments. This provides insights into evapotranspiration losses, storage changes, and hydrological efficiency.

### Analysis Components
1. Calculate runoff ratios (Q/P) at annual and seasonal scales
2. Trend analysis in runoff efficiency over time
3. Evapotranspiration proxy estimation (P - Q)
4. Water balance residuals and closure assessment
5. Climate sensitivity analysis (ΔQ/ΔP)
6. Spatial patterns in water partitioning

### Key Questions
- What proportion of precipitation becomes runoff?
- Are runoff ratios changing over time?
- How much water is lost to evapotranspiration?
- Do water balance components close within uncertainty?
- How does runoff efficiency vary spatially and seasonally?

### 6.1 Calculate Runoff Ratios and Water Balance Components

In [ ]:
# Calculate runoff ratios and water balance components
runoff_ratio_data = []

print("=" * 80)
print("RUNOFF RATIO & WATER BALANCE ANALYSIS")
print("=" * 80)
print("Calculating Q/P ratios and ET proxies...")

for gauge_id in valid_gauges:
    if gauge_id not in discharge_data or gauge_id not in precipitation_data:
        continue

    wb_stats = {"gauge_id": gauge_id}
    discharge = discharge_data[gauge_id]

    for dataset in ["era5", "mswep", "gpcp"]:
        if dataset not in precipitation_data[gauge_id]:
            continue

        prcp = precipitation_data[gauge_id][dataset]

        # Align time series
        analysis_end = "2021-09-30" if dataset == "gpcp" else "2023-12-31"
        common_idx = prcp.loc["2008":analysis_end].index.intersection(
            discharge.loc["2008":analysis_end].index
        )

        if len(common_idx) < 365:
            continue

        prcp_aligned = prcp.loc[common_idx]
        discharge_aligned = discharge.loc[common_idx]

        # Annual water balance
        prcp_annual = prcp_aligned.resample("YE").sum()
        q_annual = discharge_aligned.resample("YE").sum()

        # Ensure same years
        common_years = prcp_annual.index.intersection(q_annual.index)
        if len(common_years) < 3:
            continue

        prcp_annual = prcp_annual.loc[common_years]
        q_annual = q_annual.loc[common_years]

        # Runoff ratio (Q/P)
        qp_ratio = q_annual / prcp_annual
        qp_ratio = qp_ratio.replace([np.inf, -np.inf], np.nan)

        wb_stats[f"{dataset}_qp_ratio_mean"] = qp_ratio.mean()
        wb_stats[f"{dataset}_qp_ratio_median"] = qp_ratio.median()
        wb_stats[f"{dataset}_qp_ratio_std"] = qp_ratio.std()
        wb_stats[f"{dataset}_qp_ratio_cv"] = (
            qp_ratio.std() / qp_ratio.mean() if qp_ratio.mean() > 0 else np.nan
        )

        # Evapotranspiration proxy (P - Q)
        et_proxy = prcp_annual - q_annual
        wb_stats[f"{dataset}_et_proxy_mean"] = et_proxy.mean()
        wb_stats[f"{dataset}_et_proxy_std"] = et_proxy.std()

        # Water balance residual as fraction of P
        wb_residual = et_proxy / prcp_annual
        wb_stats[f"{dataset}_wb_residual_mean"] = wb_residual.mean()

        # Climate sensitivity (correlation between annual P and Q)
        if len(prcp_annual) >= 10:
            try:
                from scipy.stats import pearsonr

                r_pq_annual, p_pq_annual = pearsonr(
                    prcp_annual.values, q_annual.values
                )
                wb_stats[f"{dataset}_annual_pq_correlation"] = r_pq_annual

                # Regression slope (sensitivity)
                from scipy.stats import linregress

                slope, intercept, r_value, p_value, std_err = linregress(
                    prcp_annual.values, q_annual.values
                )
                wb_stats[f"{dataset}_climate_sensitivity"] = slope  # ΔQ/ΔP
            except:
                pass

        # Seasonal runoff ratios
        prcp_seasonal = prcp_aligned.to_frame("precip")
        q_seasonal = discharge_aligned.to_frame("discharge")

        prcp_seasonal["year"] = prcp_seasonal.index.year
        prcp_seasonal["season"] = prcp_seasonal.index.month.map(get_season)
        q_seasonal["year"] = q_seasonal.index.year
        q_seasonal["season"] = q_seasonal.index.month.map(get_season)

        for season in ["DJF", "MAM", "JJA", "SON"]:
            prcp_season_data = prcp_seasonal[prcp_seasonal["season"] == season]
            q_season_data = q_seasonal[q_seasonal["season"] == season]

            prcp_season_sums = prcp_season_data.groupby("year")["precip"].sum()
            q_season_sums = q_season_data.groupby("year")["discharge"].sum()

            # Align years
            common_season_years = prcp_season_sums.index.intersection(
                q_season_sums.index
            )
            if len(common_season_years) > 0:
                qp_season = (
                    q_season_sums.loc[common_season_years]
                    / prcp_season_sums.loc[common_season_years]
                )
                qp_season = qp_season.replace([np.inf, -np.inf], np.nan)
                wb_stats[f"{dataset}_{season}_qp_ratio"] = qp_season.mean()

    if len(wb_stats) > 1:
        runoff_ratio_data.append(wb_stats)

runoff_ratio_df = pd.DataFrame(runoff_ratio_data).set_index("gauge_id")

print(
    f"✓ Water balance analysis completed for {len(runoff_ratio_df)} gauges\n"
)

# Summary statistics
print("Annual Runoff Ratio (Q/P) Statistics:")
for dataset in ["era5", "mswep", "gpcp"]:
    col = f"{dataset}_qp_ratio_mean"
    if col in runoff_ratio_df.columns:
        mean_qp = runoff_ratio_df[col].mean()
        median_qp = runoff_ratio_df[col].median()
        print(
            f"  {dataset.upper():8s}: {mean_qp:.3f} (median: {median_qp:.3f})"
        )

print("\nEvapotranspiration Proxy (P - Q, mm/yr):")
for dataset in ["era5", "mswep"]:
    col = f"{dataset}_et_proxy_mean"
    if col in runoff_ratio_df.columns:
        mean_et = runoff_ratio_df[col].mean()
        print(
            f"  {dataset.upper():8s}: {mean_et:.1f} ± {runoff_ratio_df[f'{dataset}_et_proxy_std'].mean():.1f}"
        )

print("\nClimate Sensitivity (ΔQ/ΔP):")
for dataset in ["era5", "mswep"]:
    col = f"{dataset}_climate_sensitivity"
    if col in runoff_ratio_df.columns:
        mean_sens = runoff_ratio_df[col].mean()
        print(f"  {dataset.upper():8s}: {mean_sens:.3f}")

logger.info(
    f"Runoff ratio analysis completed for {len(runoff_ratio_df)} gauges"
)

In [ ]:
from scipy.stats import pearsonr

# Calculate P-Q correlations with lag analysis
pq_correlations = []

for gauge_id in valid_gauges:
    if gauge_id not in discharge_data:
        continue

    corr_stats = {"gauge_id": gauge_id}
    discharge = discharge_data[gauge_id]

    for dataset in ["era5", "mswep", "gpcp"]:
        if dataset not in precipitation_data[gauge_id]:
            continue

        prcp = precipitation_data[gauge_id][dataset]

        # Align time series (use common period for all datasets: 2008-2021 for comparability)
        analysis_end = "2021-09-30" if dataset == "gpcp" else "2023-12-31"
        common_idx = prcp.loc["2008":analysis_end].index.intersection(
            discharge.loc["2008":analysis_end].index
        )

        if len(common_idx) < 365:
            continue

        prcp_aligned = prcp.loc[common_idx]
        discharge_aligned = discharge.loc[common_idx]

        # Calculate correlations for lags 0-7 days
        lag_correlations = {}

        for lag in range(8):  # 0 to 7 days
            if lag == 0:
                # No lag - direct correlation
                try:
                    r, p = pearsonr(
                        prcp_aligned.values, discharge_aligned.values
                    )
                    lag_correlations[lag] = {"r": float(r), "p": float(p)}
                except:
                    continue
            else:
                # Shift precipitation backwards (precipitation leads discharge)
                prcp_shifted = prcp_aligned.shift(lag)
                valid_idx = prcp_shifted.notna() & discharge_aligned.notna()

                if valid_idx.sum() >= 365:
                    try:
                        r, p = pearsonr(
                            prcp_shifted[valid_idx].values,
                            discharge_aligned[valid_idx].values,
                        )
                        lag_correlations[lag] = {"r": float(r), "p": float(p)}
                    except:
                        continue

        # Store results for each lag
        for lag, result in lag_correlations.items():
            corr_stats[f"{dataset}_lag{lag}_r"] = result["r"]
            corr_stats[f"{dataset}_lag{lag}_p"] = result["p"]

        # Find best lag
        if lag_correlations:
            best_lag = max(
                lag_correlations.items(), key=lambda x: abs(x[1]["r"])
            )[0]
            corr_stats[f"{dataset}_best_lag"] = best_lag
            corr_stats[f"{dataset}_best_lag_r"] = lag_correlations[best_lag][
                "r"
            ]
            corr_stats[f"{dataset}_best_lag_p"] = lag_correlations[best_lag][
                "p"
            ]

    if len(corr_stats) > 1:
        pq_correlations.append(corr_stats)

pq_correlations_df = pd.DataFrame(pq_correlations).set_index("gauge_id")

# Summary statistics
print("=" * 80)
print("PRECIPITATION-DISCHARGE CORRELATION SUMMARY (WITH LAG ANALYSIS)")
print("=" * 80)

for dataset in ["era5", "mswep", "gpcp"]:
    col_lag0 = f"{dataset}_lag0_r"
    col_best_lag = f"{dataset}_best_lag"
    col_best_r = f"{dataset}_best_lag_r"

    if col_lag0 in pq_correlations_df.columns:
        # Lag 0 statistics
        lag0_data = pq_correlations_df[col_lag0].dropna()
        lag0_mean = lag0_data.mean()
        lag0_median = lag0_data.median()
        lag0_strong = (lag0_data > 0.5).sum()
        lag0_moderate = ((lag0_data > 0.3) & (lag0_data <= 0.5)).sum()

        # Best lag statistics
        if col_best_lag in pq_correlations_df.columns:
            best_lag_data = pq_correlations_df[col_best_lag].dropna()
            best_r_data = pq_correlations_df[col_best_r].dropna()

            mean_best_lag = best_lag_data.mean()
            median_best_lag = best_lag_data.median()
            best_r_mean = best_r_data.mean()
            best_r_median = best_r_data.median()

            improvement = best_r_data.mean() - lag0_data.mean()

            print(f"\n{dataset.upper()} Dataset:")
            print(f"  N gauges:           {len(lag0_data)}")
            print(
                f"  Lag 0 correlation:  {lag0_mean:.3f} (median: {lag0_median:.3f})"
            )
            print(
                f"    Strong (r>0.5):   {lag0_strong} ({100 * lag0_strong / len(lag0_data):.1f}%)"
            )
            print(
                f"    Moderate (0.3-0.5): {lag0_moderate} ({100 * lag0_moderate / len(lag0_data):.1f}%)"
            )
            print(
                f"  Best lag:           {mean_best_lag:.1f} days (median: {median_best_lag:.0f} days)"
            )
            print(
                f"  Best lag r:         {best_r_mean:.3f} (median: {best_r_median:.3f})"
            )
            print(
                f"  Improvement:        {improvement:.3f} (from lag optimization)"
            )

print("=" * 80)

logger.info(
    f"P-Q correlations with lag analysis calculated for {len(pq_correlations_df)} gauges"
)

## 5. Spatial Visualization and Cluster Analysis

### 5.1 Merge All Datasets and Create Spatial GeoDataFrame

In [ ]:
# Merge all datasets
combined_df = gauge_gdf.copy()

# Add precipitation statistics
combined_df = combined_df.join(prcp_stats_df, how="inner")

# Add inter-dataset correlations
combined_df = combined_df.join(inter_dataset_corr_df, how="left")

# Add precipitation trends
combined_df = combined_df.join(prcp_trends_df, how="left")

# Add temperature trends
combined_df = combined_df.join(temp_trends_df, how="left")

# Add P-Q correlations
combined_df = combined_df.join(pq_correlations_df, how="left")

# Add geographical cluster information
combined_df = combined_df.join(geo_clusters_df[["cluster_name"]], how="left")
combined_df = combined_df.rename(columns={"cluster_name": "geo_cluster_name"})

# Add latitude and longitude as explicit columns
combined_df["latitude"] = combined_df.geometry.y
combined_df["longitude"] = combined_df.geometry.x

# Add watershed area
combined_df = combined_df.join(ws_gdf[["area"]], how="left", rsuffix="_ws")

print("=" * 80)
print("COMBINED DATASET SUMMARY")
print("=" * 80)
print(f"Total gauges: {len(combined_df)}")
print(f"Total columns: {len(combined_df.columns)}")
print("\nColumn groups:")
print(
    f"  Precipitation stats:     {len([c for c in combined_df.columns if 'mean_annual' in c or 'std_annual' in c])} columns"
)
print(
    f"  Inter-dataset corr:      {len([c for c in combined_df.columns if 'era5_mswep' in c or 'era5_gpcp' in c or 'mswep_gpcp' in c])} columns"
)
print(
    f"  Precipitation trends:    {len([c for c in combined_df.columns if '_mk_trend' in c or '_sen_slope' in c])} columns"
)
print(
    f"  Temperature trends:      {len([c for c in combined_df.columns if 't_mean' in c or 't_min' in c or 't_max' in c])} columns"
)
print(
    f"  P-Q correlations:        {len([c for c in combined_df.columns if 'lag' in c and '_r' in c])} columns"
)
print("=" * 80)

logger.info(
    f"Combined dataset created: {len(combined_df)} gauges with {len(combined_df.columns)} columns"
)

### 5.2 Spatial Maps: Precipitation Trends

Visualize spatial patterns in precipitation trends for each dataset.

In [ ]:
# Create output directories
output_dir = Path("../docs/analysis_results/forcing_representation")
output_dir.mkdir(parents=True, exist_ok=True)
tables_dir = output_dir / "tables"
tables_dir.mkdir(exist_ok=True)
images_dir = output_dir / "images"
images_dir.mkdir(exist_ok=True)

# Precipitation trend spatial maps
prcp_trend_metrics = [
    "era5_sen_slope",
    "mswep_sen_slope",
    "gpcp_sen_slope",
]

prcp_trend_titles = [
    "ERA5-Land Precipitation Trend (mm/yr²)",
    "MSWEP Precipitation Trend (mm/yr²)",
    "GPCP Precipitation Trend (mm/yr²)",
]

prcp_trend_bins = {
    "era5_sen_slope": [
        (-30, -15),
        (-15, -10),
        (-10, -5),
        (-5, 5),
        (5, 10),
        (10, 15),
        (15, 30),
    ],
    "mswep_sen_slope": [
        (-30, -15),
        (-15, -10),
        (-10, -5),
        (-5, 5),
        (5, 10),
        (10, 15),
        (15, 30),
    ],
    "gpcp_sen_slope": [
        (-30, -15),
        (-15, -10),
        (-10, -5),
        (-5, 5),
        (5, 10),
        (10, 15),
        (15, 30),
    ],
}

fig_prcp_trends = russia_continuous_multiplot(
    gdf_to_plot=combined_df,
    basemap_data=basemap_data,
    metrics=prcp_trend_metrics,
    titles=prcp_trend_titles,
    main_title=f"Precipitation Trends by Dataset - {len(combined_df)} Gauges",
    bin_intervals=prcp_trend_bins,
    cmap_name="RdBu_r",
    ncols=3,
    subplot_size=(8.0, 5.0),
    with_histogram=True,
    marker_size=15,
)

fig_prcp_trends.savefig(
    images_dir / "precipitation_trends_spatial.png",
    dpi=300,
    bbox_inches="tight",
)
plt.close(fig_prcp_trends)

print(f"✓ Precipitation trend maps saved to {images_dir}")
logger.info("Precipitation trend spatial maps created")

### 5.3 Spatial Maps: Temperature Trends

In [ ]:
# Temperature trend spatial maps (using annual trends)
temp_trend_metrics = [
    "t_mean_annual_sen_slope",
    "t_min_annual_sen_slope",
    "t_max_annual_sen_slope",
]

temp_trend_titles = [
    "Mean Temperature Trend - Annual (°C/yr)",
    "Min Temperature Trend - Annual (°C/yr)",
    "Max Temperature Trend - Annual (°C/yr)",
]

temp_trend_bins = {
    "t_mean_annual_sen_slope": [
        (-0.02, 0.00),
        (0.00, 0.02),
        (0.02, 0.04),
        (0.04, 0.06),
        (0.06, 0.08),
        (0.08, 0.10),
        (0.10, 0.15),
    ],
    "t_min_annual_sen_slope": [
        (-0.02, 0.00),
        (0.00, 0.02),
        (0.02, 0.04),
        (0.04, 0.06),
        (0.06, 0.08),
        (0.08, 0.10),
        (0.10, 0.15),
    ],
    "t_max_annual_sen_slope": [
        (-0.02, 0.00),
        (0.00, 0.02),
        (0.02, 0.04),
        (0.04, 0.06),
        (0.06, 0.08),
        (0.08, 0.10),
        (0.10, 0.15),
    ],
}

fig_temp_trends = russia_continuous_multiplot(
    gdf_to_plot=combined_df,
    basemap_data=basemap_data,
    metrics=temp_trend_metrics,
    titles=temp_trend_titles,
    main_title=f"Temperature Trends (ERA5-Land, 2008-2023) - {len(combined_df)} Gauges",
    bin_intervals=temp_trend_bins,
    cmap_name="YlOrRd",
    ncols=3,
    subplot_size=(8.0, 5.0),
    with_histogram=True,
    marker_size=15,
)

fig_temp_trends.savefig(
    images_dir / "temperature_trends_spatial.png", dpi=300, bbox_inches="tight"
)
plt.close(fig_temp_trends)

print(f"✓ Temperature trend maps saved to {images_dir}")
logger.info("Temperature trend spatial maps created")


In [ ]:
# Temperature trend spatial maps - Seasonal (Mean Temperature)
temp_seasonal_metrics = [
    "t_mean_winter_sen_slope",
    "t_mean_spring_sen_slope",
    "t_mean_summer_sen_slope",
    "t_mean_autumn_sen_slope",
]

temp_seasonal_titles = [
    "Winter Temperature Trend (°C/yr)",
    "Spring Temperature Trend (°C/yr)",
    "Summer Temperature Trend (°C/yr)",
    "Autumn Temperature Trend (°C/yr)",
]

temp_seasonal_bins = {
    "t_mean_winter_sen_slope": [
        (-0.05, 0.00),
        (0.00, 0.05),
        (0.05, 0.10),
        (0.10, 0.15),
        (0.15, 0.20),
        (0.20, 0.25),
        (0.25, 0.35),
    ],
    "t_mean_spring_sen_slope": [
        (-0.05, 0.00),
        (0.00, 0.05),
        (0.05, 0.10),
        (0.10, 0.15),
        (0.15, 0.20),
        (0.20, 0.25),
        (0.25, 0.35),
    ],
    "t_mean_summer_sen_slope": [
        (-0.05, 0.00),
        (0.00, 0.05),
        (0.05, 0.10),
        (0.10, 0.15),
        (0.15, 0.20),
        (0.20, 0.25),
        (0.25, 0.35),
    ],
    "t_mean_autumn_sen_slope": [
        (-0.05, 0.00),
        (0.00, 0.05),
        (0.05, 0.10),
        (0.10, 0.15),
        (0.15, 0.20),
        (0.20, 0.25),
        (0.25, 0.35),
    ],
}

fig_temp_seasonal = russia_continuous_multiplot(
    gdf_to_plot=combined_df,
    basemap_data=basemap_data,
    metrics=temp_seasonal_metrics,
    titles=temp_seasonal_titles,
    main_title=f"Seasonal Temperature Trends (ERA5-Land, 2008-2023) - {len(combined_df)} Gauges",
    bin_intervals=temp_seasonal_bins,
    cmap_name="RdYlBu_r",
    ncols=2,
    subplot_size=(10.0, 6.0),
    with_histogram=True,
    marker_size=10,
)

fig_temp_seasonal.savefig(
    images_dir / "temperature_trends_seasonal.png",
    dpi=300,
    bbox_inches="tight",
)
plt.close(fig_temp_seasonal)

print(f"✓ Seasonal temperature trend maps saved to {images_dir}")
logger.info("Seasonal temperature trend spatial maps created")
display(fig_temp_seasonal)


### 5.4 Spatial Maps: Precipitation-Discharge Correlations and Lag Effects

In [ ]:
# P-Q correlation spatial maps
pq_corr_metrics = [
    "era5_lag0_r",
    "mswep_lag0_r",
    "gpcp_lag0_r",
    "era5_best_lag_r",
    "mswep_best_lag_r",
    "gpcp_best_lag_r",
]

pq_corr_titles = [
    "ERA5 P-Q Correlation (lag 0)",
    "MSWEP P-Q Correlation (lag 0)",
    "GPCP P-Q Correlation (lag 0)",
    "ERA5 P-Q Best Lag Correlation",
    "MSWEP P-Q Best Lag Correlation",
    "GPCP P-Q Best Lag Correlation",
]

pq_corr_bins = {
    "era5_lag0_r": [
        (-0.2, 0.0),
        (0.0, 0.2),
        (0.2, 0.3),
        (0.3, 0.4),
        (0.4, 0.5),
        (0.5, 0.6),
        (0.6, 0.8),
    ],
    "mswep_lag0_r": [
        (-0.2, 0.0),
        (0.0, 0.2),
        (0.2, 0.3),
        (0.3, 0.4),
        (0.4, 0.5),
        (0.5, 0.6),
        (0.6, 0.8),
    ],
    "gpcp_lag0_r": [
        (-0.2, 0.0),
        (0.0, 0.2),
        (0.2, 0.3),
        (0.3, 0.4),
        (0.4, 0.5),
        (0.5, 0.6),
        (0.6, 0.8),
    ],
    "era5_best_lag_r": [
        (-0.2, 0.0),
        (0.0, 0.2),
        (0.2, 0.3),
        (0.3, 0.4),
        (0.4, 0.5),
        (0.5, 0.6),
        (0.6, 0.8),
    ],
    "mswep_best_lag_r": [
        (-0.2, 0.0),
        (0.0, 0.2),
        (0.2, 0.3),
        (0.3, 0.4),
        (0.4, 0.5),
        (0.5, 0.6),
        (0.6, 0.8),
    ],
    "gpcp_best_lag_r": [
        (-0.2, 0.0),
        (0.0, 0.2),
        (0.2, 0.3),
        (0.3, 0.4),
        (0.4, 0.5),
        (0.5, 0.6),
        (0.6, 0.8),
    ],
}

fig_pq_corr = russia_continuous_multiplot(
    gdf_to_plot=combined_df,
    basemap_data=basemap_data,
    metrics=pq_corr_metrics,
    titles=pq_corr_titles,
    main_title=f"Precipitation-Discharge Correlations by Dataset - {len(combined_df)} Gauges",
    bin_intervals=pq_corr_bins,
    cmap_name="RdYlBu_r",
    ncols=3,
    subplot_size=(8.0, 5.0),
    with_histogram=True,
    marker_size=15,
)

fig_pq_corr.savefig(
    images_dir / "pq_correlations_spatial.png", dpi=300, bbox_inches="tight"
)
plt.close(fig_pq_corr)

print(f"✓ P-Q correlation maps saved to {images_dir}")
logger.info("P-Q correlation spatial maps created")

### 5.5 Spatial Maps: Best Lag Distribution

In [ ]:
# Lag distribution spatial maps
lag_metrics = [
    "era5_best_lag",
    "mswep_best_lag",
    "gpcp_best_lag",
]

lag_titles = [
    "ERA5 Optimal Lag (days)",
    "MSWEP Optimal Lag (days)",
    "GPCP Optimal Lag (days)",
]

lag_bins = {
    "era5_best_lag": [(0, 1), (1, 2), (2, 3), (3, 4), (4, 5), (5, 6), (6, 7)],
    "mswep_best_lag": [(0, 1), (1, 2), (2, 3), (3, 4), (4, 5), (5, 6), (6, 7)],
    "gpcp_best_lag": [(0, 1), (1, 2), (2, 3), (3, 4), (4, 5), (5, 6), (6, 7)],
}

fig_lag = russia_continuous_multiplot(
    gdf_to_plot=combined_df,
    basemap_data=basemap_data,
    metrics=lag_metrics,
    titles=lag_titles,
    main_title=f"Optimal Precipitation-Discharge Lag by Dataset - {len(combined_df)} Gauges",
    bin_intervals=lag_bins,
    cmap_name="viridis",
    ncols=3,
    subplot_size=(8.0, 5.0),
    with_histogram=True,
    marker_size=15,
)

fig_lag.savefig(
    images_dir / "pq_lag_spatial.png", dpi=300, bbox_inches="tight"
)
display(fig_lag)
print(f"✓ Lag distribution maps saved to {images_dir}")
logger.info("Lag distribution spatial maps created")

### 5.6 Discharge-Precipitation Relationship Assessment Plots

Comprehensive visualization of the discharge-precipitation relationship across different datasets, lags, and spatial clusters.

In [ ]:
# Plot 1: Correlation distribution by dataset (boxplots)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(
    "Discharge-Precipitation Correlation Distributions by Dataset",
    fontsize=16,
    fontweight="bold",
)

# Lag 0 correlations
ax = axes[0, 0]
data_lag0 = [
    pq_correlations_df["era5_lag0_r"].dropna(),
    pq_correlations_df["mswep_lag0_r"].dropna(),
    pq_correlations_df["gpcp_lag0_r"].dropna(),
]
bp = ax.boxplot(
    data_lag0,
    labels=["ERA5", "MSWEP", "GPCP"],
    patch_artist=True,
    showmeans=True,
)
for patch in bp["boxes"]:
    patch.set_facecolor("#7fcdbb")
ax.set_ylabel("Pearson Correlation", fontsize=11)
ax.set_title("Lag 0 (Same Day)", fontsize=12, fontweight="bold")
ax.grid(True, alpha=0.3, axis="y")
ax.axhline(0, color="red", linestyle="--", alpha=0.5, linewidth=1)

# Best lag correlations
ax = axes[0, 1]
data_best = [
    pq_correlations_df["era5_best_lag_r"].dropna(),
    pq_correlations_df["mswep_best_lag_r"].dropna(),
    pq_correlations_df["gpcp_best_lag_r"].dropna(),
]
bp = ax.boxplot(
    data_best,
    labels=["ERA5", "MSWEP", "GPCP"],
    patch_artist=True,
    showmeans=True,
)
for patch in bp["boxes"]:
    patch.set_facecolor("#41b6c4")
ax.set_ylabel("Pearson Correlation", fontsize=11)
ax.set_title("Best Lag (0-7 days)", fontsize=12, fontweight="bold")
ax.grid(True, alpha=0.3, axis="y")
ax.axhline(0, color="red", linestyle="--", alpha=0.5, linewidth=1)

# Correlation improvement (best - lag0)
ax = axes[1, 0]
improvements = {
    "ERA5": (
        pq_correlations_df["era5_best_lag_r"]
        - pq_correlations_df["era5_lag0_r"]
    ).dropna(),
    "MSWEP": (
        pq_correlations_df["mswep_best_lag_r"]
        - pq_correlations_df["mswep_lag0_r"]
    ).dropna(),
    "GPCP": (
        pq_correlations_df["gpcp_best_lag_r"]
        - pq_correlations_df["gpcp_lag0_r"]
    ).dropna(),
}
bp = ax.boxplot(
    improvements.values(),
    labels=improvements.keys(),
    patch_artist=True,
    showmeans=True,
)
for patch in bp["boxes"]:
    patch.set_facecolor("#fec44f")
ax.set_ylabel("Correlation Improvement (Δr)", fontsize=11)
ax.set_title("Optimal Lag Improvement", fontsize=12, fontweight="bold")
ax.grid(True, alpha=0.3, axis="y")
ax.axhline(0, color="black", linestyle="--", alpha=0.5, linewidth=1)

# Optimal lag frequency distribution
ax = axes[1, 1]
era5_lags = pq_correlations_df["era5_best_lag"].dropna()
lag_counts = era5_lags.value_counts().sort_index()
bars = ax.bar(lag_counts.index, lag_counts.values, color="#225ea8", alpha=0.8)
ax.set_xlabel("Lag (days)", fontsize=11)
ax.set_ylabel("Number of Gauges", fontsize=11)
ax.set_title("ERA5 Optimal Lag Distribution", fontsize=12, fontweight="bold")
ax.set_xticks(range(8))
ax.grid(True, alpha=0.3, axis="y")

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2.0,
        height,
        f"{int(height)}",
        ha="center",
        va="bottom",
        fontsize=9,
    )

plt.tight_layout()

output_path = images_dir / "pq_correlation_distributions.png"
plt.savefig(output_path, dpi=300, bbox_inches="tight")
plt.show()

logger.info(f"P-Q correlation distribution plots saved to {output_path}")

In [ ]:
# Plot 2: Correlation vs geographical clusters (boxplots by cluster)
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle(
    "P-Q Correlations by Geographical Cluster (ERA5-Land)",
    fontsize=16,
    fontweight="bold",
)

# Merge correlation data with cluster information
# combined_df has gauge_id as index and geo_cluster_name column
pq_with_clusters = combined_df[["geo_cluster_name"]].merge(
    pq_correlations_df,
    left_index=True,
    right_index=True,
    how="inner",
)

# Lag 0 by cluster
ax = axes[0]
cluster_data_lag0 = [
    pq_with_clusters[pq_with_clusters["geo_cluster_name"] == cluster][
        "era5_lag0_r"
    ].dropna()
    for cluster in sorted(
        pq_with_clusters["geo_cluster_name"].dropna().unique()
    )
]
cluster_labels = sorted(pq_with_clusters["geo_cluster_name"].dropna().unique())

bp = ax.boxplot(
    cluster_data_lag0,
    labels=cluster_labels,
    patch_artist=True,
    showmeans=True,
)
for patch in bp["boxes"]:
    patch.set_facecolor("#7fcdbb")
ax.set_xlabel("Geographical Cluster", fontsize=11)
ax.set_ylabel("Pearson Correlation", fontsize=11)
ax.set_title("Lag 0 Correlation", fontsize=12, fontweight="bold")
ax.grid(True, alpha=0.3, axis="y")
ax.tick_params(axis="x", rotation=45, labelsize=9)

# Best lag by cluster
ax = axes[1]
cluster_data_best = [
    pq_with_clusters[pq_with_clusters["geo_cluster_name"] == cluster][
        "era5_best_lag_r"
    ].dropna()
    for cluster in sorted(
        pq_with_clusters["geo_cluster_name"].dropna().unique()
    )
]

bp = ax.boxplot(
    cluster_data_best,
    labels=cluster_labels,
    patch_artist=True,
    showmeans=True,
)
for patch in bp["boxes"]:
    patch.set_facecolor("#41b6c4")
ax.set_xlabel("Geographical Cluster", fontsize=11)
ax.set_ylabel("Pearson Correlation", fontsize=11)
ax.set_title("Best Lag Correlation", fontsize=12, fontweight="bold")
ax.grid(True, alpha=0.3, axis="y")
ax.tick_params(axis="x", rotation=45, labelsize=9)

# Optimal lag distribution by cluster
ax = axes[2]
cluster_data_lag = [
    pq_with_clusters[pq_with_clusters["geo_cluster_name"] == cluster][
        "era5_best_lag"
    ].dropna()
    for cluster in sorted(
        pq_with_clusters["geo_cluster_name"].dropna().unique()
    )
]

bp = ax.boxplot(
    cluster_data_lag,
    labels=cluster_labels,
    patch_artist=True,
    showmeans=True,
)
for patch in bp["boxes"]:
    patch.set_facecolor("#fec44f")
ax.set_xlabel("Geographical Cluster", fontsize=11)
ax.set_ylabel("Optimal Lag (days)", fontsize=11)
ax.set_title("Optimal Lag Distribution", fontsize=12, fontweight="bold")
ax.grid(True, alpha=0.3, axis="y")
ax.tick_params(axis="x", rotation=45, labelsize=9)

plt.tight_layout()

output_path = images_dir / "pq_correlation_by_cluster.png"
plt.savefig(output_path, dpi=300, bbox_inches="tight")
plt.show()

logger.info(f"P-Q correlation cluster analysis plots saved to {output_path}")

In [ ]:
# Plot 3: Example time series showing P-Q relationships at different lags
# Select 4 example gauges with different correlation patterns

# Sort by ERA5 best lag correlation to find examples
sorted_pq = pq_correlations_df.sort_values("era5_best_lag_r", ascending=False)

# Select 4 examples: high correlation, medium, low, and best improvement
improvement_series = sorted_pq["era5_best_lag_r"] - sorted_pq["era5_lag0_r"]
examples = [
    sorted_pq.index[0],  # Highest correlation
    sorted_pq.index[len(sorted_pq) // 3],  # Medium-high
    sorted_pq.index[2 * len(sorted_pq) // 3],  # Medium-low
    improvement_series.idxmax(),  # Best improvement from lagging
]

fig, axes = plt.subplots(4, 1, figsize=(16, 14))
fig.suptitle(
    "Example Discharge-Precipitation Relationships (ERA5-Land)",
    fontsize=16,
    fontweight="bold",
    y=0.995,
)

for idx, gauge_id in enumerate(examples):
    ax = axes[idx]

    # Get data
    if "era5" not in precipitation_data.get(gauge_id, {}):
        continue

    discharge = discharge_data[gauge_id]
    prcp = precipitation_data[gauge_id]["era5"]

    # Get correlation info
    best_lag = int(pq_correlations_df.loc[gauge_id, "era5_best_lag"])
    lag0_corr = pq_correlations_df.loc[gauge_id, "era5_lag0_r"]
    best_corr = pq_correlations_df.loc[gauge_id, "era5_best_lag_r"]
    improvement = best_corr - lag0_corr

    # Align data for 2020-2021 (sample period)
    plot_start = "2020-01-01"
    plot_end = "2021-12-31"

    discharge_plot = discharge.loc[plot_start:plot_end]
    prcp_plot = prcp.loc[plot_start:plot_end]

    # Normalize for visualization
    discharge_norm = (discharge_plot - discharge_plot.min()) / (
        discharge_plot.max() - discharge_plot.min()
    )
    prcp_norm = (prcp_plot - prcp_plot.min()) / (
        prcp_plot.max() - prcp_plot.min()
    )

    # Plot precipitation as bars (inverted y-axis for visual effect)
    ax2 = ax.twinx()
    ax2.bar(
        prcp_norm.index,
        prcp_norm.values,
        color="#4575b4",
        alpha=0.5,
        width=1,
        label="Precipitation (normalized)",
    )
    ax2.set_ylim(0, 1.5)
    ax2.set_ylabel("Precipitation (normalized)", fontsize=10, color="#4575b4")
    ax2.tick_params(axis="y", labelcolor="#4575b4")

    # Plot discharge as line
    ax.plot(
        discharge_norm.index,
        discharge_norm.values,
        color="#d73027",
        linewidth=2,
        label="Discharge (normalized)",
    )
    ax.set_ylim(0, 1.2)
    ax.set_ylabel("Discharge (normalized)", fontsize=10, color="#d73027")
    ax.tick_params(axis="y", labelcolor="#d73027")
    ax.grid(True, alpha=0.3)

    # Title with correlation info
    title_text = (
        f"Gauge {gauge_id} | "
        f"r(lag 0) = {lag0_corr:.3f}, r(lag {best_lag}) = {best_corr:.3f}, "
        f"Δr = {improvement:+.3f}"
    )
    ax.set_title(title_text, fontsize=11, pad=10)

    if idx == 3:
        ax.set_xlabel("Date", fontsize=11)

plt.tight_layout()

output_path = images_dir / "pq_relationship_examples.png"
plt.savefig(output_path, dpi=300, bbox_inches="tight")
plt.show()

logger.info(f"P-Q relationship example time series saved to {output_path}")

### 5.6 Cluster-Based Analysis

Summarize results by geographical clusters.

In [ ]:
# Merge new analysis results into combined_df for cluster analysis
combined_df_extended = combined_df.copy()

# Merge precipitation sums
if "prcp_sums_df" in globals():
    # Select key columns from prcp_sums_df
    prcp_sum_cols = [
        "era5_annual_sum_mean",
        "mswep_annual_sum_mean",
        "gpcp_annual_sum_mean",
        "era5_djf_sum_mean",
        "era5_mam_sum_mean",
        "era5_jja_sum_mean",
        "era5_son_sum_mean",
    ]
    available_prcp_cols = [
        col for col in prcp_sum_cols if col in prcp_sums_df.columns
    ]
    combined_df_extended = combined_df_extended.join(
        prcp_sums_df[available_prcp_cols], how="left"
    )

# Merge runoff ratios
if "runoff_ratio_df" in globals():
    # Select key columns from runoff_ratio_df
    runoff_cols = [
        "era5_qp_annual_mean",
        "mswep_qp_annual_mean",
        "era5_et_proxy_mean",
        "mswep_et_proxy_mean",
        "era5_climate_sensitivity",
        "mswep_climate_sensitivity",
    ]
    available_runoff_cols = [
        col for col in runoff_cols if col in runoff_ratio_df.columns
    ]
    combined_df_extended = combined_df_extended.join(
        runoff_ratio_df[available_runoff_cols], how="left"
    )

# Merge precipitation sum trends
if "prcp_sum_trends_df" in globals():
    # Select key trend columns
    trend_cols = [
        "era5_annual_sum_sen_slope",
        "mswep_annual_sum_sen_slope",
        "gpcp_annual_sum_sen_slope",
    ]
    available_trend_cols = [
        col for col in trend_cols if col in prcp_sum_trends_df.columns
    ]
    combined_df_extended = combined_df_extended.join(
        prcp_sum_trends_df[available_trend_cols], how="left"
    )

# Cluster-based summary statistics (EXPANDED with new metrics)
agg_dict = {
    # Precipitation statistics (daily means)
    "era5_mean_annual": ["mean", "std", "count"],
    "mswep_mean_annual": ["mean", "std"],
    "gpcp_mean_annual": ["mean", "std"],
    # Precipitation trends (daily)
    "era5_sen_slope": ["mean", "median"],
    "mswep_sen_slope": ["mean", "median"],
    "gpcp_sen_slope": ["mean", "median"],
    # Temperature trends (annual)
    "t_mean_annual_sen_slope": ["mean", "median"],
    "t_min_annual_sen_slope": ["mean", "median"],
    "t_max_annual_sen_slope": ["mean", "median"],
    # P-Q correlations (lag 0)
    "era5_lag0_r": ["mean", "median", "std"],
    "mswep_lag0_r": ["mean", "median", "std"],
    "gpcp_lag0_r": ["mean", "median", "std"],
    # Best lag
    "era5_best_lag": ["mean", "median"],
    "mswep_best_lag": ["mean", "median"],
    "gpcp_best_lag": ["mean", "median"],
    # Best lag correlation
    "era5_best_lag_r": ["mean", "median"],
    "mswep_best_lag_r": ["mean", "median"],
    "gpcp_best_lag_r": ["mean", "median"],
}

# Add new precipitation sum metrics if available
if "era5_annual_sum_mean" in combined_df_extended.columns:
    agg_dict["era5_annual_sum_mean"] = ["mean", "std"]
    agg_dict["mswep_annual_sum_mean"] = ["mean", "std"]
    agg_dict["gpcp_annual_sum_mean"] = ["mean", "std"]

# Add seasonal precipitation sums if available
seasonal_cols = [
    "era5_djf_sum_mean",
    "era5_mam_sum_mean",
    "era5_jja_sum_mean",
    "era5_son_sum_mean",
]
for col in seasonal_cols:
    if col in combined_df_extended.columns:
        agg_dict[col] = ["mean", "std"]

# Add runoff ratio metrics if available
if "era5_qp_annual_mean" in combined_df_extended.columns:
    agg_dict["era5_qp_annual_mean"] = ["mean", "median", "std"]
    agg_dict["mswep_qp_annual_mean"] = ["mean", "median", "std"]

# Add ET proxy if available
if "era5_et_proxy_mean" in combined_df_extended.columns:
    agg_dict["era5_et_proxy_mean"] = ["mean", "std"]
    agg_dict["mswep_et_proxy_mean"] = ["mean", "std"]

# Add climate sensitivity if available
if "era5_climate_sensitivity" in combined_df_extended.columns:
    agg_dict["era5_climate_sensitivity"] = ["mean", "std"]
    agg_dict["mswep_climate_sensitivity"] = ["mean", "std"]

# Add precipitation sum trends if available
if "era5_annual_sum_sen_slope" in combined_df_extended.columns:
    agg_dict["era5_annual_sum_sen_slope"] = ["mean", "median"]
    agg_dict["mswep_annual_sum_sen_slope"] = ["mean", "median"]
    agg_dict["gpcp_annual_sum_sen_slope"] = ["mean", "median"]

cluster_summary = combined_df_extended.groupby("geo_cluster_name").agg(
    agg_dict
)

# Flatten column names
cluster_summary.columns = [
    "_".join(col).strip() for col in cluster_summary.columns
]

# Save to CSV
cluster_summary.to_csv(tables_dir / "cluster_summary_forcing_analysis.csv")

print("=" * 80)
print("CLUSTER-BASED SUMMARY (UPDATED WITH NEW METRICS)")
print("=" * 80)
print(
    f"Analyzed {combined_df_extended['geo_cluster_name'].nunique()} geographical clusters"
)
print(f"Total metrics per cluster: {len(cluster_summary.columns)}")
print("\nNew metrics added:")
if "era5_annual_sum_mean" in combined_df_extended.columns:
    print("  ✓ Annual precipitation sums")
if "era5_qp_annual_mean" in combined_df_extended.columns:
    print("  ✓ Runoff ratios (Q/P)")
if "era5_et_proxy_mean" in combined_df_extended.columns:
    print("  ✓ ET proxy (P-Q)")
if "era5_climate_sensitivity" in combined_df_extended.columns:
    print("  ✓ Climate sensitivity")
if "era5_annual_sum_sen_slope" in combined_df_extended.columns:
    print("  ✓ Precipitation sum trends")
print("\nSample statistics (first 3 clusters, selected columns):")
sample_cols = [
    col
    for col in cluster_summary.columns[:10]
    if any(
        x in col for x in ["era5_mean", "era5_qp", "era5_annual_sum", "count"]
    )
]
if sample_cols:
    print(cluster_summary[sample_cols].head(3).to_string())
print("=" * 80)
print(
    f"\n✓ Cluster summary saved to {tables_dir / 'cluster_summary_forcing_analysis.csv'}"
)

logger.info(
    f"Cluster-based summary created for {combined_df_extended['geo_cluster_name'].nunique()} clusters with {len(cluster_summary.columns)} metrics"
)

## 6. Export Data and Create Summary Tables

### 6.1 Export Complete Dataset

In [ ]:
# Export all analysis results to tables directory
print("=" * 80)
print("EXPORTING ANALYSIS RESULTS")
print("=" * 80)

# 1. Export precipitation sums analysis
if "prcp_sums_df" in globals():
    prcp_sums_output = tables_dir / "precipitation_sums_analysis.csv"
    prcp_sums_df.to_csv(prcp_sums_output)
    print(f"✓ Exported precipitation sums: {prcp_sums_output.name}")
    print(f"  - Shape: {prcp_sums_df.shape}")
    print(f"  - Columns: {len(prcp_sums_df.columns)}")

# 2. Export precipitation sum trends
if "prcp_sum_trends_df" in globals():
    prcp_trends_output = tables_dir / "precipitation_sum_trends.csv"
    prcp_sum_trends_df.to_csv(prcp_trends_output)
    print(f"✓ Exported precipitation sum trends: {prcp_trends_output.name}")
    print(f"  - Shape: {prcp_sum_trends_df.shape}")

# 3. Export runoff ratio analysis
if "runoff_ratio_df" in globals():
    runoff_output = tables_dir / "runoff_ratio_analysis.csv"
    runoff_ratio_df.to_csv(runoff_output)
    print(f"✓ Exported runoff ratios: {runoff_output.name}")
    print(f"  - Shape: {runoff_ratio_df.shape}")
    print(f"  - Columns: {len(runoff_ratio_df.columns)}")

# 4. Update comprehensive forcing analysis file with new columns
if all(
    df in globals()
    for df in ["prcp_sums_df", "runoff_ratio_df", "combined_df_extended"]
):
    # Merge all analyses
    comprehensive_df = combined_df_extended.copy()

    # Save comprehensive analysis
    comprehensive_csv = tables_dir / "forcing_analysis_complete_v2.csv"
    comprehensive_gpkg = output_dir / "forcing_analysis_complete_v2.gpkg"

    # Save CSV (without geometry)
    if "geometry" in comprehensive_df.columns:
        comprehensive_df.drop(columns=["geometry"]).to_csv(comprehensive_csv)
    else:
        comprehensive_df.to_csv(comprehensive_csv)

    # Save GeoPackage (with geometry)
    if "geometry" in comprehensive_df.columns:
        comprehensive_df.to_file(comprehensive_gpkg, driver="GPKG")
        print(
            f"✓ Exported comprehensive analysis (v2): {comprehensive_csv.name}"
        )
        print(f"  - CSV rows: {len(comprehensive_df)}")
        print(f"  - Total columns: {len(comprehensive_df.columns)}")
        print(
            f"✓ Exported comprehensive GeoPackage: {comprehensive_gpkg.name}"
        )
    else:
        print(
            "⚠ No geometry column found, skipping GeoPackage export for comprehensive analysis"
        )

print("=" * 80)
print("EXPORT COMPLETE")
print("=" * 80)

logger.info(
    "All analysis results exported successfully to tables and geopackage files"
)

## 7. Spatial Visualizations for New Analyses

### 7.1 Precipitation Sum Trends Spatial Maps

In [ ]:
# Create spatial maps for precipitation sum trends
from src.plots.continuous_maps import russia_continuous_multiplot

# Prepare data for mapping
if "prcp_sum_trends_df" in globals() and "gauge_gdf" in globals():
    # Merge with geometry for mapping
    prcp_trends_combined = combined_df_extended[
        combined_df_extended.index.isin(prcp_sum_trends_df.index)
    ].copy()

    # Define metrics and titles
    trend_metrics = [
        "era5_annual_sum_sen_slope",
        "mswep_annual_sum_sen_slope",
        "gpcp_annual_sum_sen_slope",
    ]

    trend_titles = [
        "ERA5-Land Annual Sum Trends (mm/yr²)",
        "MSWEP Annual Sum Trends (mm/yr²)",
        "GPCP Annual Sum Trends (mm/yr²)",
    ]

    # Define symmetric bins around zero
    trend_bins = {
        "era5_annual_sum_sen_slope": [
            (-10, -5),
            (-5, -2),
            (-2, 0),
            (0, 2),
            (2, 5),
            (5, 10),
        ],
        "mswep_annual_sum_sen_slope": [
            (-10, -5),
            (-5, -2),
            (-2, 0),
            (0, 2),
            (2, 5),
            (5, 10),
        ],
        "gpcp_annual_sum_sen_slope": [
            (-10, -5),
            (-5, -2),
            (-2, 0),
            (0, 2),
            (2, 5),
            (5, 10),
        ],
    }

    fig_prcp_sum_trends = russia_continuous_multiplot(
        gdf_to_plot=prcp_trends_combined,
        basemap_data=basemap_data,
        metrics=trend_metrics,
        titles=trend_titles,
        main_title=f"Annual Precipitation Sum Trends by Dataset - {len(prcp_trends_combined)} Gauges",
        bin_intervals=trend_bins,
        cmap_name="RdBu_r",
        ncols=3,
        subplot_size=(8.0, 5.0),
        with_histogram=True,
        marker_size=15,
    )

    # Save figure
    output_file = images_dir / "precipitation_sums_trends_spatial.png"
    fig_prcp_sum_trends.savefig(output_file, dpi=300, bbox_inches="tight")
    print(f"✓ Saved precipitation sum trends map: {output_file.name}")

    display(fig_prcp_sum_trends)

    logger.info("Precipitation sum trends spatial maps created")
else:
    print("⚠ Required data not available for precipitation sum trends mapping")

### 7.2 Runoff Ratio Spatial Patterns

In [ ]:
# Create spatial maps for runoff ratios
if "runoff_ratio_df" in globals() and "ws_gdf" in globals():
    # Merge runoff_ratio_df with geometry from combined_df which has geometry
    runoff_gdf = combined_df[["geometry"]].join(runoff_ratio_df, how="inner")

    # Figure 1: Runoff Ratios (Q/P)
    qp_metrics = ["era5_qp_ratio_mean", "mswep_qp_ratio_mean"]
    qp_titles = [
        "ERA5-Land Runoff Ratio (Q/P)",
        "MSWEP Runoff Ratio (Q/P)",
    ]
    qp_bins = {
        "era5_qp_ratio_mean": [
            (0, 0.1),
            (0.1, 0.2),
            (0.2, 0.4),
            (0.4, 0.6),
            (0.6, 0.8),
            (0.8, 1.0),
        ],
        "mswep_qp_ratio_mean": [
            (0, 0.1),
            (0.1, 0.2),
            (0.2, 0.4),
            (0.4, 0.6),
            (0.6, 0.8),
            (0.8, 1.0),
        ],
    }

    fig_qp = russia_continuous_multiplot(
        gdf_to_plot=runoff_gdf,
        basemap_data=basemap_data,
        metrics=qp_metrics,
        titles=qp_titles,
        main_title=f"Annual Runoff Ratio (Q/P) - {len(runoff_gdf)} Gauges",
        bin_intervals=qp_bins,
        cmap_name="YlGnBu",
        ncols=2,
        subplot_size=(8.0, 6.0),
        with_histogram=True,
        marker_size=15,
    )

    output_file = images_dir / "runoff_ratio_qp_spatial.png"
    fig_qp.savefig(output_file, dpi=300, bbox_inches="tight")
    print(f"✓ Saved runoff ratio (Q/P) map: {output_file.name}")
    display(fig_qp)

    # Figure 2: ET Proxy and Climate Sensitivity
    wb_metrics = ["era5_et_proxy_mean", "era5_climate_sensitivity"]
    wb_titles = [
        "ERA5-Land ET Proxy (P - Q, mm/yr)",
        "ERA5-Land Climate Sensitivity (ΔQ/ΔP)",
    ]
    wb_bins = {
        "era5_et_proxy_mean": [
            (0, 100),
            (100, 200),
            (200, 300),
            (300, 400),
            (400, 500),
            (500, 600),
        ],
        "era5_climate_sensitivity": [
            (0, 0.2),
            (0.2, 0.4),
            (0.4, 0.6),
            (0.6, 0.8),
            (0.8, 1.0),
            (1.0, 1.5),
        ],
    }

    fig_wb = russia_continuous_multiplot(
        gdf_to_plot=runoff_gdf,
        basemap_data=basemap_data,
        metrics=wb_metrics,
        titles=wb_titles,
        main_title=f"Water Balance Components - {len(runoff_gdf)} Gauges",
        bin_intervals=wb_bins,
        cmap_name="YlOrRd",
        ncols=2,
        subplot_size=(8.0, 6.0),
        with_histogram=True,
        marker_size=15,
    )

    output_file = images_dir / "water_balance_patterns.png"
    fig_wb.savefig(output_file, dpi=300, bbox_inches="tight")
    print(f"✓ Saved water balance patterns map: {output_file.name}")
    display(fig_wb)

    logger.info("Runoff ratio and water balance spatial maps created")
else:
    print("⚠ Required data not available for runoff ratio mapping")

### 7.3 Seasonal Precipitation and Water Balance Boxplots

In [ ]:
# Create boxplots for seasonal analysis
if "prcp_sums_df" in globals() and "runoff_ratio_df" in globals():
    # Figure 1: Seasonal Precipitation Sums
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    seasons = ["DJF", "MAM", "JJA", "SON"]
    season_labels = [
        "DJF\n(Winter)",
        "MAM\n(Spring)",
        "JJA\n(Summer)",
        "SON\n(Autumn)",
    ]

    # ERA5 seasonal sums
    era5_seasonal_data = [
        prcp_sums_df[f"era5_{s}_sum_mean"].dropna() for s in seasons
    ]
    bp1 = axes[0].boxplot(
        era5_seasonal_data,
        labels=season_labels,
        patch_artist=True,
        showfliers=False,
    )
    axes[0].set_title(
        "ERA5-Land Seasonal Precipitation Sums", fontsize=14, fontweight="bold"
    )
    axes[0].set_ylabel("Precipitation Sum (mm/season)", fontsize=12)
    axes[0].grid(axis="y", alpha=0.3)
    for patch in bp1["boxes"]:
        patch.set_facecolor("steelblue")

    # MSWEP seasonal sums
    mswep_seasonal_data = [
        prcp_sums_df[f"mswep_{s}_sum_mean"].dropna() for s in seasons
    ]
    bp2 = axes[1].boxplot(
        mswep_seasonal_data,
        labels=season_labels,
        patch_artist=True,
        showfliers=False,
    )
    axes[1].set_title(
        "MSWEP Seasonal Precipitation Sums", fontsize=14, fontweight="bold"
    )
    axes[1].set_ylabel("Precipitation Sum (mm/season)", fontsize=12)
    axes[1].grid(axis="y", alpha=0.3)
    for patch in bp2["boxes"]:
        patch.set_facecolor("darkgreen")

    # GPCP seasonal sums
    gpcp_seasonal_data = [
        prcp_sums_df[f"gpcp_{s}_sum_mean"].dropna() for s in seasons
    ]
    bp3 = axes[2].boxplot(
        gpcp_seasonal_data,
        labels=season_labels,
        patch_artist=True,
        showfliers=False,
    )
    axes[2].set_title(
        "GPCP Seasonal Precipitation Sums", fontsize=14, fontweight="bold"
    )
    axes[2].set_ylabel("Precipitation Sum (mm/season)", fontsize=12)
    axes[2].grid(axis="y", alpha=0.3)
    for patch in bp3["boxes"]:
        patch.set_facecolor("darkorange")

    plt.tight_layout()
    output_file = images_dir / "seasonal_precipitation_sums_boxplots.png"
    plt.savefig(output_file, dpi=300, bbox_inches="tight")
    print(f"✓ Saved seasonal precipitation boxplots: {output_file.name}")
    plt.show()

    # Figure 2: Seasonal Runoff Ratios
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # ERA5 seasonal Q/P
    era5_qp_seasonal_data = [
        runoff_ratio_df[f"era5_{s}_qp_ratio"].dropna() for s in seasons
    ]
    bp4 = axes[0].boxplot(
        era5_qp_seasonal_data,
        labels=season_labels,
        patch_artist=True,
        showfliers=False,
    )
    axes[0].set_title(
        "ERA5-Land Seasonal Runoff Ratios", fontsize=14, fontweight="bold"
    )
    axes[0].set_ylabel("Runoff Ratio (Q/P)", fontsize=12)
    axes[0].set_ylim(0, 1.2)
    axes[0].axhline(
        y=1.0, color="red", linestyle="--", alpha=0.5, label="Q/P = 1"
    )
    axes[0].grid(axis="y", alpha=0.3)
    axes[0].legend()
    for patch in bp4["boxes"]:
        patch.set_facecolor("steelblue")

    # MSWEP seasonal Q/P
    mswep_qp_seasonal_data = [
        runoff_ratio_df[f"mswep_{s}_qp_ratio"].dropna() for s in seasons
    ]
    bp5 = axes[1].boxplot(
        mswep_qp_seasonal_data,
        labels=season_labels,
        patch_artist=True,
        showfliers=False,
    )
    axes[1].set_title(
        "MSWEP Seasonal Runoff Ratios", fontsize=14, fontweight="bold"
    )
    axes[1].set_ylabel("Runoff Ratio (Q/P)", fontsize=12)
    axes[1].set_ylim(0, 1.2)
    axes[1].axhline(
        y=1.0, color="red", linestyle="--", alpha=0.5, label="Q/P = 1"
    )
    axes[1].grid(axis="y", alpha=0.3)
    axes[1].legend()
    for patch in bp5["boxes"]:
        patch.set_facecolor("darkgreen")

    plt.tight_layout()
    output_file = images_dir / "seasonal_runoff_ratios_boxplots.png"
    plt.savefig(output_file, dpi=300, bbox_inches="tight")
    print(f"✓ Saved seasonal runoff ratio boxplots: {output_file.name}")
    plt.show()

    logger.info(
        "Seasonal boxplots created for precipitation sums and runoff ratios"
    )
else:
    print("⚠ Required data not available for seasonal boxplots")

---

## 8. Event-Based P-Q Response Analysis

**Objective**: Analyze individual precipitation events and discharge responses to understand event-scale hydrological behavior.

**Methods**:
- Precipitation event detection using threshold-based approach
- Peak discharge timing and lag analysis
- Event response efficiency (event Q/P ratio)
- Recession analysis and baseflow separation
- Dataset comparison for event representation

**Key Metrics**:
- Event frequency and magnitude
- Mean lag between P peak and Q peak
- Event runoff coefficient
- Recession constants
- Event detection success rate by dataset

In [ ]:
"""
Event-Based P-Q Response Analysis
Detect precipitation events and analyze discharge responses
"""
from scipy.signal import find_peaks
from scipy.stats import linregress

# Event detection parameters
P_EVENT_THRESHOLD = 10.0  # mm/day - minimum precipitation to qualify as event
MIN_EVENT_SPACING = 3  # days - minimum time between events
Q_RESPONSE_WINDOW = (
    14  # days - window to search for discharge peak after P event
)

logger.info("Starting event-based P-Q response analysis")
logger.info(
    f"Event threshold: {P_EVENT_THRESHOLD} mm/day, "
    f"Min spacing: {MIN_EVENT_SPACING} days, "
    f"Q response window: {Q_RESPONSE_WINDOW} days"
)

# Storage for event analysis results
event_results = []

# Process each gauge
for gauge_id in valid_gauges:
    try:
        # Load datasets
        datasets_loaded = {}

        # ERA5
        era5_path = era5_dir / f"{gauge_id}.csv"
        if era5_path.exists():
            era5_df = pd.read_csv(era5_path, index_col=0, parse_dates=True)
            if "prcp" in era5_df.columns:
                datasets_loaded["era5"] = era5_df["prcp"]

        # MSWEP
        mswep_path = mswep_dir / f"{gauge_id}.csv"
        if mswep_path.exists():
            mswep_df = pd.read_csv(mswep_path, index_col=0, parse_dates=True)
            if "prcp" in mswep_df.columns:
                datasets_loaded["mswep"] = mswep_df["prcp"]

        # Discharge
        discharge_path = discharge_dir_full / f"{gauge_id}.csv"
        if discharge_path.exists():
            discharge_df = pd.read_csv(
                discharge_path, index_col=0, parse_dates=True
            )
            # Try different possible column names for discharge
            if "q" in discharge_df.columns:
                discharge_series = discharge_df["q"]
            elif "q_mm_day" in discharge_df.columns:
                discharge_series = discharge_df["q_mm_day"]
            elif "q_cms" in discharge_df.columns:
                discharge_series = discharge_df["q_cms"]
            else:
                continue
        else:
            continue

        # Ensure datetime index
        if not isinstance(discharge_series.index, pd.DatetimeIndex):
            discharge_series.index = pd.to_datetime(discharge_series.index)

        # Process each precipitation dataset
        for dataset_name, prcp_series in datasets_loaded.items():
            # Ensure datetime index
            if not isinstance(prcp_series.index, pd.DatetimeIndex):
                prcp_series.index = pd.to_datetime(prcp_series.index)

            # Align precipitation and discharge
            common_idx = prcp_series.index.intersection(discharge_series.index)
            if len(common_idx) < 365:  # Need at least 1 year
                continue

            prcp_aligned = prcp_series.loc[common_idx].sort_index()
            discharge_aligned = discharge_series.loc[common_idx].sort_index()

            # Detect precipitation events using peak finding
            prcp_values = prcp_aligned.values
            event_indices, event_properties = find_peaks(
                prcp_values,
                height=P_EVENT_THRESHOLD,
                distance=MIN_EVENT_SPACING,
            )

            if len(event_indices) == 0:
                continue

            # Analyze each event
            event_lags = []
            event_qp_ratios = []
            event_q_responses = []

            for event_idx in event_indices:
                event_date = common_idx[event_idx]
                event_prcp = prcp_values[event_idx]

                # Define response window
                response_end_idx = min(
                    event_idx + Q_RESPONSE_WINDOW, len(discharge_aligned)
                )

                if response_end_idx <= event_idx:
                    continue

                # Find discharge peak in response window
                q_window = discharge_aligned.iloc[event_idx:response_end_idx]
                q_peak_relative_idx = q_window.values.argmax()
                q_peak_idx = event_idx + q_peak_relative_idx
                q_peak_value = discharge_aligned.iloc[q_peak_idx]

                # Calculate lag (days)
                lag_days = q_peak_relative_idx

                # Calculate event runoff coefficient
                # Sum discharge over response window and compare to event P
                q_sum = discharge_aligned.iloc[
                    event_idx:response_end_idx
                ].sum()
                event_qp = q_sum / event_prcp if event_prcp > 0 else np.nan

                event_lags.append(lag_days)
                event_qp_ratios.append(event_qp)
                event_q_responses.append(q_peak_value)

            # Calculate event statistics
            if len(event_lags) > 0:
                event_results.append(
                    {
                        "gauge_id": gauge_id,
                        "dataset": dataset_name,
                        "n_events": len(event_indices),
                        "event_frequency": len(event_indices)
                        / (len(common_idx) / 365.25),  # events/year
                        "mean_event_prcp": np.mean(prcp_values[event_indices]),
                        "mean_event_lag": np.mean(event_lags),
                        "median_event_lag": np.median(event_lags),
                        "mean_event_qp": np.nanmean(event_qp_ratios),
                        "median_event_qp": np.nanmedian(event_qp_ratios),
                        "mean_q_response": np.mean(event_q_responses),
                        "std_event_lag": np.std(event_lags),
                    }
                )

    except Exception as e:
        logger.error(f"Error processing gauge {gauge_id}: {e}")
        continue

# Create DataFrame with proper handling of empty results
if len(event_results) > 0:
    event_response_df = pd.DataFrame(event_results)

    # Summary statistics
    logger.info("\nEvent Response Analysis Summary:")
    logger.info(
        f"Total gauges analyzed: {event_response_df['gauge_id'].nunique()}"
    )
    logger.info(f"Total datasets: {event_response_df['dataset'].nunique()}")
    logger.info("\nEvent Statistics:")
    logger.info(
        f"Mean event frequency: {event_response_df['event_frequency'].mean():.1f} events/year"
    )
    logger.info(
        f"Mean event lag: {event_response_df['mean_event_lag'].mean():.1f} days"
    )
    logger.info(
        f"Mean event Q/P ratio: {event_response_df['mean_event_qp'].mean():.3f}"
    )

    # Dataset comparison
    dataset_comparison = (
        event_response_df.groupby("dataset")
        .agg(
            {
                "event_frequency": ["mean", "std"],
                "mean_event_lag": ["mean", "std"],
                "mean_event_qp": ["mean", "std"],
            }
        )
        .round(3)
    )

    logger.info(f"\nDataset Comparison:\n{dataset_comparison}")

    display(event_response_df.head(10))
else:
    logger.warning(
        "No events detected for any gauge. "
        f"Consider adjusting parameters: P_EVENT_THRESHOLD={P_EVENT_THRESHOLD}, "
        f"MIN_EVENT_SPACING={MIN_EVENT_SPACING}, Q_RESPONSE_WINDOW={Q_RESPONSE_WINDOW}"
    )
    # Create empty DataFrame with expected columns
    event_response_df = pd.DataFrame(
        columns=[
            "gauge_id",
            "dataset",
            "n_events",
            "event_frequency",
            "mean_event_prcp",
            "mean_event_lag",
            "median_event_lag",
            "mean_event_qp",
            "median_event_qp",
            "mean_q_response",
            "std_event_lag",
        ]
    )
    logger.info("Created empty event_response_df with expected schema")


In [ ]:
# Temperature trend spatial maps (using annual trends)
temp_trend_metrics = [
    "t_mean_annual_sen_slope",
    "t_min_annual_sen_slope",
    "t_max_annual_sen_slope",
]

temp_trend_titles = [
    "Mean Temperature Trend - Annual (°C/yr)",
    "Min Temperature Trend - Annual (°C/yr)",
    "Max Temperature Trend - Annual (°C/yr)",
]

temp_trend_bins = {
    "t_mean_annual_sen_slope": [
        (-0.02, 0.00),
        (0.00, 0.02),
        (0.02, 0.04),
        (0.04, 0.06),
        (0.06, 0.08),
        (0.08, 0.10),
        (0.10, 0.15),
    ],
    "t_min_annual_sen_slope": [
        (-0.02, 0.00),
        (0.00, 0.02),
        (0.02, 0.04),
        (0.04, 0.06),
        (0.06, 0.08),
        (0.08, 0.10),
        (0.10, 0.15),
    ],
    "t_max_annual_sen_slope": [
        (-0.02, 0.00),
        (0.00, 0.02),
        (0.02, 0.04),
        (0.04, 0.06),
        (0.06, 0.08),
        (0.08, 0.10),
        (0.10, 0.15),
    ],
}

fig_temp_trends = russia_continuous_multiplot(
    gdf_to_plot=combined_df,
    basemap_data=basemap_data,
    metrics=temp_trend_metrics,
    titles=temp_trend_titles,
    main_title=f"Temperature Trends (ERA5-Land, 2008-2023) - {len(combined_df)} Gauges",
    bin_intervals=temp_trend_bins,
    cmap_name="YlOrRd",
    ncols=3,
    subplot_size=(8.0, 5.0),
    with_histogram=True,
    marker_size=15,
)

fig_temp_trends.savefig(
    images_dir / "temperature_trends_spatial.png", dpi=300, bbox_inches="tight"
)
display(fig_temp_trends)

print(f"✓ Temperature trend maps saved to {images_dir}")
logger.info("Temperature trend spatial maps created")


---

## 9. Seasonal Regime Classification

**Objective**: Classify catchments by dominant hydrological regime (snowmelt vs rainfall) and analyze seasonal P-Q efficiency.

**Methods**:
- Snowmelt regime identification via seasonal Q/P patterns
- Rainfall regime classification via summer-autumn P-Q correlation
- Winter accumulation index (DJF P sum / Annual P)
- Spring release efficiency (MAM Q / DJF P)
- Regime-based catchment grouping

**Key Metrics**:
- Regime classification (snowmelt-dominated, rainfall-dominated, mixed)
- Winter accumulation fraction
- Spring melt efficiency
- Season-specific P-Q correlations
- Regime spatial distribution

### 9.1 Regime Identification and Seasonal Efficiency

In [ ]:
"""
Seasonal Regime Classification
Identify snowmelt vs rainfall-dominated catchments
"""
logger.info("Starting seasonal regime classification")

# Regime classification thresholds
SNOWMELT_QP_THRESHOLD = 0.4  # MAM Q/P > 0.4 indicates snowmelt dominance
WINTER_ACCUMULATION_THRESHOLD = 0.2  # DJF P / Annual P > 0.2
SPRING_EFFICIENCY_THRESHOLD = 0.5  # MAM Q / DJF P > 0.5

# Storage for regime analysis
regime_results = []

# Process each gauge
for gauge_id in valid_gauges:
    try:
        # Load datasets
        datasets_loaded = {}

        # ERA5
        era5_path = era5_dir / f"{gauge_id}.csv"
        if era5_path.exists():
            era5_df = pd.read_csv(era5_path, index_col=0, parse_dates=True)
            if "prcp" in era5_df.columns:
                datasets_loaded["era5"] = era5_df["prcp"]

        # MSWEP
        mswep_path = mswep_dir / f"{gauge_id}.csv"
        if mswep_path.exists():
            mswep_df = pd.read_csv(mswep_path, index_col=0, parse_dates=True)
            if "prcp" in mswep_df.columns:
                datasets_loaded["mswep"] = mswep_df["prcp"]

        # Discharge
        discharge_path = discharge_dir_full / f"{gauge_id}.csv"
        if discharge_path.exists():
            discharge_df = pd.read_csv(
                discharge_path, index_col=0, parse_dates=True
            )
            # Try different possible column names for discharge
            if "q" in discharge_df.columns:
                discharge_series = discharge_df["q"]
            elif "q_mm_day" in discharge_df.columns:
                discharge_series = discharge_df["q_mm_day"]
            elif "q_cms" in discharge_df.columns:
                discharge_series = discharge_df["q_cms"]
            else:
                continue
        else:
            continue

        # Ensure datetime index
        if not isinstance(discharge_series.index, pd.DatetimeIndex):
            discharge_series.index = pd.to_datetime(discharge_series.index)

        # Process each precipitation dataset
        for dataset_name, prcp_series in datasets_loaded.items():
            # Ensure datetime index
            if not isinstance(prcp_series.index, pd.DatetimeIndex):
                prcp_series.index = pd.to_datetime(prcp_series.index)

            # Align data
            common_idx = prcp_series.index.intersection(discharge_series.index)
            if len(common_idx) < 365:
                continue

            prcp_aligned = prcp_series.loc[common_idx].sort_index()
            discharge_aligned = discharge_series.loc[common_idx].sort_index()

            # Add season column
            prcp_df = pd.DataFrame({"prcp": prcp_aligned})
            prcp_df["season"] = prcp_df.index.month.map(
                {
                    12: "DJF",
                    1: "DJF",
                    2: "DJF",
                    3: "MAM",
                    4: "MAM",
                    5: "MAM",
                    6: "JJA",
                    7: "JJA",
                    8: "JJA",
                    9: "SON",
                    10: "SON",
                    11: "SON",
                }
            )

            discharge_df = pd.DataFrame({"q": discharge_aligned})
            discharge_df["season"] = discharge_df.index.month.map(
                {
                    12: "DJF",
                    1: "DJF",
                    2: "DJF",
                    3: "MAM",
                    4: "MAM",
                    5: "MAM",
                    6: "JJA",
                    7: "JJA",
                    8: "JJA",
                    9: "SON",
                    10: "SON",
                    11: "SON",
                }
            )

            # Calculate annual and seasonal totals
            prcp_df["year"] = prcp_df.index.year
            discharge_df["year"] = discharge_df.index.year

            # Annual sums
            annual_p = prcp_df.groupby("year")["prcp"].sum().mean()
            annual_q = discharge_df.groupby("year")["q"].sum().mean()

            # Seasonal sums (multi-year average)
            seasonal_p = prcp_df.groupby("season")["prcp"].sum().mean()
            seasonal_q = discharge_df.groupby("season")["q"].sum().mean()

            # Winter accumulation fraction
            djf_p = (
                seasonal_p.get("DJF", 0) if hasattr(seasonal_p, "get") else 0
            )
            winter_accum_frac = djf_p / annual_p if annual_p > 0 else 0

            # Spring discharge fraction
            mam_q = (
                seasonal_q.get("MAM", 0) if hasattr(seasonal_q, "get") else 0
            )
            spring_q_frac = mam_q / annual_q if annual_q > 0 else 0

            # Spring efficiency (MAM Q / DJF P)
            spring_efficiency = mam_q / djf_p if djf_p > 0 else 0

            # Spring runoff ratio (MAM Q/P)
            mam_p = (
                seasonal_p.get("MAM", 0) if hasattr(seasonal_p, "get") else 0
            )
            spring_qp = mam_q / mam_p if mam_p > 0 else 0

            # Summer runoff ratio (JJA Q/P)
            jja_p = (
                seasonal_p.get("JJA", 0) if hasattr(seasonal_p, "get") else 0
            )
            jja_q = (
                seasonal_q.get("JJA", 0) if hasattr(seasonal_q, "get") else 0
            )
            summer_qp = jja_q / jja_p if jja_p > 0 else 0

            # Season-specific P-Q correlations
            seasonal_correlations = {}
            for season in ["DJF", "MAM", "JJA", "SON"]:
                season_prcp = prcp_df[prcp_df["season"] == season]["prcp"]
                season_q = discharge_df[discharge_df["season"] == season]["q"]

                if len(season_prcp) > 30 and len(season_q) > 30:
                    # Align by index
                    common_season_idx = season_prcp.index.intersection(
                        season_q.index
                    )
                    if len(common_season_idx) > 30:
                        r, p = pearsonr(
                            season_prcp.loc[common_season_idx],
                            season_q.loc[common_season_idx],
                        )
                        seasonal_correlations[season] = r
                    else:
                        seasonal_correlations[season] = np.nan
                else:
                    seasonal_correlations[season] = np.nan

            # Regime classification logic
            # Snowmelt-dominated: High spring Q/P, high winter accumulation, high spring efficiency
            snowmelt_score = (
                (spring_qp > SNOWMELT_QP_THRESHOLD)
                + (winter_accum_frac > WINTER_ACCUMULATION_THRESHOLD)
                + (spring_efficiency > SPRING_EFFICIENCY_THRESHOLD)
            )

            # Rainfall-dominated: Higher summer correlation, lower spring dominance
            rainfall_score = (
                seasonal_correlations.get("JJA", 0)
                > seasonal_correlations.get("MAM", 0)
            ) + (summer_qp > spring_qp / 2)

            if snowmelt_score >= 2:
                regime = "snowmelt"
            elif rainfall_score >= 1 and snowmelt_score < 2:
                regime = "rainfall"
            else:
                regime = "mixed"

            regime_results.append(
                {
                    "gauge_id": gauge_id,
                    "dataset": dataset_name,
                    "regime": regime,
                    "winter_accum_frac": winter_accum_frac,
                    "spring_q_frac": spring_q_frac,
                    "spring_efficiency": spring_efficiency,
                    "spring_qp": spring_qp,
                    "summer_qp": summer_qp,
                    "djf_corr": seasonal_correlations.get("DJF", np.nan),
                    "mam_corr": seasonal_correlations.get("MAM", np.nan),
                    "jja_corr": seasonal_correlations.get("JJA", np.nan),
                    "son_corr": seasonal_correlations.get("SON", np.nan),
                    "snowmelt_score": snowmelt_score,
                    "rainfall_score": rainfall_score,
                }
            )

    except Exception as e:
        logger.error(f"Error processing gauge {gauge_id}: {e}")
        continue

# Create DataFrame with proper handling of empty results
if len(regime_results) > 0:
    seasonal_regime_df = pd.DataFrame(regime_results)

    # Summary statistics
    logger.info("\nSeasonal Regime Classification Summary:")
    logger.info(
        f"Total gauges analyzed: {seasonal_regime_df['gauge_id'].nunique()}"
    )

    regime_counts = (
        seasonal_regime_df.groupby(["dataset", "regime"])
        .size()
        .unstack(fill_value=0)
    )
    logger.info(f"\nRegime Distribution by Dataset:\n{regime_counts}")

    # Average metrics by regime
    regime_metrics = (
        seasonal_regime_df.groupby("regime")
        .agg(
            {
                "winter_accum_frac": "mean",
                "spring_efficiency": "mean",
                "spring_qp": "mean",
                "summer_qp": "mean",
                "mam_corr": "mean",
                "jja_corr": "mean",
            }
        )
        .round(3)
    )

    logger.info(f"\nAverage Metrics by Regime:\n{regime_metrics}")

    display(seasonal_regime_df.head(10))
else:
    logger.warning(
        "No seasonal regime classifications completed for any gauge. "
        "Check data availability and alignment requirements (min 365 days)."
    )
    # Create empty DataFrame with expected columns
    seasonal_regime_df = pd.DataFrame(
        columns=[
            "gauge_id",
            "dataset",
            "regime",
            "winter_accum_frac",
            "spring_q_frac",
            "spring_efficiency",
            "spring_qp",
            "summer_qp",
            "djf_corr",
            "mam_corr",
            "jja_corr",
            "son_corr",
            "snowmelt_score",
            "rainfall_score",
        ]
    )
    logger.info("Created empty seasonal_regime_df with expected schema")


In [ ]:
# Table 1: Precipitation dataset comparison
dataset_comparison = pd.DataFrame(
    {
        "Dataset": ["ERA5-Land", "MSWEP", "GPCP"],
        "N Gauges": [
            len(dataset_availability["era5"]),
            len(dataset_availability["mswep"]),
            len(dataset_availability["gpcp"]),
        ],
        "Coverage (%)": [
            100 * len(dataset_availability["era5"]) / len(valid_gauges),
            100 * len(dataset_availability["mswep"]) / len(valid_gauges),
            100 * len(dataset_availability["gpcp"]) / len(valid_gauges),
        ],
        "Mean Annual P (mm/yr)": [
            prcp_stats_df["era5_mean_annual"].mean(),
            prcp_stats_df["mswep_mean_annual"].mean(),
            prcp_stats_df["gpcp_mean_annual"].mean(),
        ],
        "Std Annual P (mm/yr)": [
            prcp_stats_df["era5_mean_annual"].std(),
            prcp_stats_df["mswep_mean_annual"].std(),
            prcp_stats_df["gpcp_mean_annual"].std(),
        ],
        "Mean P-Q Corr (lag 0)": [
            pq_correlations_df["era5_lag0_r"].mean(),
            pq_correlations_df["mswep_lag0_r"].mean(),
            pq_correlations_df["gpcp_lag0_r"].mean(),
        ],
        "Mean Best Lag (days)": [
            pq_correlations_df["era5_best_lag"].mean(),
            pq_correlations_df["mswep_best_lag"].mean(),
            pq_correlations_df["gpcp_best_lag"].mean(),
        ],
        "Mean Best Lag Corr": [
            pq_correlations_df["era5_best_lag_r"].mean(),
            pq_correlations_df["mswep_best_lag_r"].mean(),
            pq_correlations_df["gpcp_best_lag_r"].mean(),
        ],
    }
)

dataset_comparison.to_csv(
    tables_dir / "precipitation_dataset_comparison.csv", index=False
)
print("✓ Table 1: Precipitation dataset comparison")
print(dataset_comparison.to_string(index=False))
print()

# Table 2: Trend summary
trend_summary = []

for dataset in ["era5", "mswep", "gpcp"]:
    mk_col = f"{dataset}_mk_trend"
    pval_col = f"{dataset}_mk_p_value"
    slope_col = f"{dataset}_sen_slope"

    if mk_col in prcp_trends_df.columns:
        total = prcp_trends_df[mk_col].notna().sum()
        significant = (prcp_trends_df[pval_col] < 0.05).sum()
        # Use string comparison for trend direction
        increasing = (
            (prcp_trends_df[pval_col] < 0.05)
            & (prcp_trends_df[mk_col] == "increasing")
        ).sum()
        decreasing = (
            (prcp_trends_df[pval_col] < 0.05)
            & (prcp_trends_df[mk_col] == "decreasing")
        ).sum()

        trend_summary.append(
            {
                "Variable": f"Precipitation ({dataset.upper()})",
                "N Gauges": total,
                "Significant (%)": f"{100 * significant / total:.1f}",
                "Increasing (%)": f"{100 * increasing / total:.1f}",
                "Decreasing (%)": f"{100 * decreasing / total:.1f}",
                "Mean Slope": f"{prcp_trends_df[slope_col].mean():.2f} mm/yr²",
            }
        )

for var, label in [
    ("t_mean", "Temperature (Mean)"),
    ("t_min", "Temperature (Min)"),
    ("t_max", "Temperature (Max)"),
]:
    mk_col = f"{var}_mk_trend"
    pval_col = f"{var}_mk_p_value"
    slope_col = f"{var}_sen_slope"

    if mk_col in temp_trends_df.columns:
        total = temp_trends_df[mk_col].notna().sum()
        significant = (temp_trends_df[pval_col] < 0.05).sum()
        # Use string comparison for trend direction
        warming = (
            (temp_trends_df[pval_col] < 0.05)
            & (temp_trends_df[mk_col] == "increasing")
        ).sum()
        cooling = (
            (temp_trends_df[pval_col] < 0.05)
            & (temp_trends_df[mk_col] == "decreasing")
        ).sum()

        trend_summary.append(
            {
                "Variable": label,
                "N Gauges": total,
                "Significant (%)": f"{100 * significant / total:.1f}",
                "Increasing (%)": f"{100 * warming / total:.1f}",
                "Decreasing (%)": f"{100 * cooling / total:.1f}",
                "Mean Slope": f"{temp_trends_df[slope_col].mean():.4f} °C/yr",
            }
        )

trend_summary_df = pd.DataFrame(trend_summary)
trend_summary_df.to_csv(tables_dir / "trend_summary.csv", index=False)
print("✓ Table 2: Trend summary")
print(trend_summary_df.to_string(index=False))
print()

# Table 3: Inter-dataset correlations
inter_corr_summary = pd.DataFrame(
    {
        "Comparison": ["ERA5 vs MSWEP", "ERA5 vs GPCP", "MSWEP vs GPCP"],
        "Mean Correlation": [
            inter_dataset_corr_df["era5_mswep_r"].mean()
            if "era5_mswep_r" in inter_dataset_corr_df.columns
            else np.nan,
            inter_dataset_corr_df["era5_gpcp_r"].mean()
            if "era5_gpcp_r" in inter_dataset_corr_df.columns
            else np.nan,
            inter_dataset_corr_df["mswep_gpcp_r"].mean()
            if "mswep_gpcp_r" in inter_dataset_corr_df.columns
            else np.nan,
        ],
        "Std Correlation": [
            inter_dataset_corr_df["era5_mswep_r"].std()
            if "era5_mswep_r" in inter_dataset_corr_df.columns
            else np.nan,
            inter_dataset_corr_df["era5_gpcp_r"].std()
            if "era5_gpcp_r" in inter_dataset_corr_df.columns
            else np.nan,
            inter_dataset_corr_df["mswep_gpcp_r"].std()
            if "mswep_gpcp_r" in inter_dataset_corr_df.columns
            else np.nan,
        ],
        "Mean Bias (mm/day)": [
            inter_dataset_corr_df["era5_mswep_bias"].mean()
            if "era5_mswep_bias" in inter_dataset_corr_df.columns
            else np.nan,
            inter_dataset_corr_df["era5_gpcp_bias"].mean()
            if "era5_gpcp_bias" in inter_dataset_corr_df.columns
            else np.nan,
            inter_dataset_corr_df["mswep_gpcp_bias"].mean()
            if "mswep_gpcp_bias" in inter_dataset_corr_df.columns
            else np.nan,
        ],
    }
)

inter_corr_summary.to_csv(
    tables_dir / "inter_dataset_correlations.csv", index=False
)
print("✓ Table 3: Inter-dataset correlations")
print(inter_corr_summary.to_string(index=False))
print()

print("=" * 80)
print("ALL SUMMARY TABLES SAVED")
print("=" * 80)

logger.info("Summary tables created and saved")
